# Family A / Run 4 -- gold cache build. Exploratory, DO NOT SUBMIT.

Reuses V13's own pick_slots/order_slices/read_slot cells verbatim against real DICOMs, then caches raw uint8 slot tensors (not frozen pooled features) for a trainable Family A encoder. No 24-member ensemble, no submission-shaped output.


In [ ]:
"""Private, exploratory gold-cohort adapter. Never a leaderboard submission."""
import hashlib
import json
import os
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd

GOLD_UID = 'StudyInstanceUID'
GOLD_TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
                'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


def gold_write(path, data):
    Path(path).write_text(json.dumps(data, indent=2, allow_nan=False) + '\n', encoding='utf-8')


def prepare_gold(real, work, temporary_parent=None):
    """Expose fully labeled studies as test, without reports or labels in test inputs."""
    real, work = Path(real), Path(work)
    work.mkdir(parents=True, exist_ok=True)
    if (work / 'cohort_audit.json').exists():
        raise RuntimeError('Existing validation artifacts: use a fresh session/work directory')
    tr = pd.read_csv(real / 'train.csv', dtype={GOLD_UID: str})
    if tr[GOLD_UID].isna().any() or tr[GOLD_UID].duplicated().any():
        raise ValueError('Invalid training study identities')
    gold = tr.loc[tr[GOLD_TARGETS].notna().all(axis=1), [GOLD_UID] + GOLD_TARGETS].copy()
    if len(gold) < 20 or not gold[GOLD_TARGETS].isin([0, 1]).all().all():
        raise ValueError('Expected at least 20 fully expert-labeled binary studies')
    if any(gold[t].nunique() != 2 for t in GOLD_TARGETS):
        raise ValueError('Every target requires both classes')
    ids = gold[GOLD_UID].tolist()
    series = pd.read_csv(real / 'train_series.csv', dtype={GOLD_UID: str, 'SeriesInstanceUID': str})
    selected = series.loc[series[GOLD_UID].isin(ids)].copy()
    if set(selected[GOLD_UID]) != set(ids):
        raise ValueError('Gold study missing series metadata')
    for uid in ids:
        if '/' in uid or '\\' in uid or uid in ('.', '..'):
            raise ValueError('Unsafe study identifier')
        if not (real / 'train_series' / uid).is_dir():
            raise FileNotFoundError('Gold study image folder absent: ' + uid)
    if temporary_parent is not None:
        Path(temporary_parent).mkdir(parents=True, exist_ok=True)
    pseudo_parent = Path(tempfile.mkdtemp(prefix='v14-gold-', dir=temporary_parent))
    pseudo = pseudo_parent / 'rsna-knee-abnormality-detection'
    (pseudo / 'test_series').mkdir(parents=True)
    gold[[GOLD_UID]].to_csv(pseudo / 'test.csv', index=False)
    selected.to_csv(pseudo / 'test_series.csv', index=False)
    sample = gold[[GOLD_UID]].assign(**{t: .5 for t in GOLD_TARGETS})
    sample.to_csv(pseudo / 'sample_submission.csv', index=False)
    # Stage 1 uses training row count to size its cache, not for inference labels.
    tr[[GOLD_UID]].to_csv(pseudo / 'train.csv', index=False)
    for uid in ids:
        os.symlink((real / 'train_series' / uid).resolve(), pseudo / 'test_series' / uid,
                   target_is_directory=True)
    gold.to_csv(work / 'gold_labels.csv', index=False)
    gold[[GOLD_UID]].to_csv(work / 'gold_study_ids.csv', index=False)
    audit = {
        'status': 'EXPLORATORY_NOT_INDEPENDENT_CONFIRMATION', 'studies': len(gold),
        'cohort_definition': 'All train.csv rows with all 12 expert targets present; no outcome filtering.',
        'source_train_csv_sha256': hashlib.sha256((real / 'train.csv').read_bytes()).hexdigest(),
        'labels_sha256': hashlib.sha256((work / 'gold_labels.csv').read_bytes()).hexdigest(),
        'test_inputs_contain_labels_or_reports': False, 'training_enabled': False,
        'patient_grouping_verified': False, 'bootstrap_unit': 'study (patient grouping unavailable)',
        'coatnet_source_audit': {
            'source': 'https://www.kaggle.com/code/dreaddevelopment/knee-mri-training-the-twelve-finding-model',
            'gradient_exclusion': 'Author source excludes gold_ids from train_ids.',
            'checkpoint_selection': 'Gold AUC selects epochs/top-k/SWA; GOLD IS NOT UNTOUCHED.',
            'actual_checkpoint_id_manifests': 'Not independently verified by source alone.'},
        'other_families': 'Training/calibration exposure UNKNOWN unless explicit loaded metadata establishes it.',
        'plus_0_02_verified': False, 'leaderboard_submission_allowed': False,
        'counts': {t: {'positive': int(gold[t].sum()), 'negative': int((gold[t] == 0).sum())}
                   for t in GOLD_TARGETS},
    }
    gold_write(work / 'cohort_audit.json', audit)
    print('GOLD AUDIT BEFORE INFERENCE:', json.dumps(audit), flush=True)
    return pseudo, work


def audit_metadata(payload, gold_ids):
    """Read explicit checkpoint metadata only; never infer exclusion from a fold number."""
    result = {'explicit_id_sets': [], 'metadata': {}}
    gold_ids = set(map(str, gold_ids))
    id_keys = {'train_ids', 'training_ids', 'train_uids', 'training_uids', 'val_ids',
               'valid_ids', 'validation_ids', 'val_uids', 'gold_ids', 'test_ids'}
    scalar_keys = {'epoch', 'gold_auc', 'auc', 'best_auc', 'n_train', 'n_gold', 'fold',
                   'version', 'tag', 'seed', 'training_studies', 'validation_studies'}

    def visit(value, prefix='', depth=0):
        if depth > 4 or not isinstance(value, dict):
            return
        for key, item in value.items():
            if not isinstance(key, str):
                continue
            name = prefix + key
            if key.lower() in id_keys and isinstance(item, (list, tuple, set, np.ndarray)):
                overlap = sorted(gold_ids.intersection(map(str, item)))
                result['explicit_id_sets'].append({'key': name, 'count': len(item),
                                                   'gold_overlap_count': len(overlap),
                                                   'gold_overlap_ids': overlap})
            elif key in scalar_keys and isinstance(item, (str, int, float, bool, type(None))):
                if not isinstance(item, float) or np.isfinite(item):
                    result['metadata'][name] = item
            elif key in ('meta', 'metadata', 'config', 'cfg', 'manifest', 'provenance'):
                visit(item, name + '.', depth + 1)
            elif key == 'folds' and isinstance(item, list):
                for i, record in enumerate(item):
                    visit(record, name + f'.{i}.', depth + 1)
    visit(payload)
    result['exclusion_verified'] = False  # Absence of IDs is never evidence of exclusion.
    return result


def install_checkpoint_audit(work, ids):
    """Observe the normal loads; no second GPU/model load and no changed tensors."""
    import torch
    import threading
    original = torch.load
    records, lock = [], threading.Lock()

    def observed(*args, **kwargs):
        payload = original(*args, **kwargs)
        path = str(args[0] if args else kwargs.get('f', 'unknown'))
        record = {'path': path, **audit_metadata(payload, ids)}
        with lock:
            records.append(record)
            gold_write(Path(work) / 'loaded_checkpoint_audit.json', {
                'scope': 'Observed metadata only; no missing-ID exclusion inference.',
                'independent_confirmation_eligible': False, 'records': records})
        return payload
    torch.load = observed
    return records


def evaluate_gold(run, ablations, diagnostics):
    """All results remain descriptive/exploratory, including positive deltas."""
    labels = pd.read_csv(run.work / 'gold_labels.csv', dtype={GOLD_UID: str})
    baseline = pd.read_csv(run.primary, dtype={GOLD_UID: str})
    labels = diagnostics.align(labels, run.ids, labels=True)
    baseline = diagnostics.align(baseline, run.ids)
    y = labels[GOLD_TARGETS].to_numpy(float)
    baseline_aucs = diagnostics.aucs(y, baseline[GOLD_TARGETS].to_numpy(float))
    result = {'status': 'EXPLORATORY_GOLD_SELECTED_NOT_CONFIRMATION',
              'studies': len(labels), 'macro_auc': float(baseline_aucs.mean()),
              'plus_0_02_verified': False, 'leaderboard_improvement_measured': False,
              'ci_caveat': 'Study-level, no verified patient grouping; selected gold set; multiple candidates unadjusted.',
              'baseline': {}, 'candidates': {}, 'stage_macro_auc': {}}
    for j, t in enumerate(GOLD_TARGETS):
        result['baseline'][t] = {'auc': float(baseline_aucs[j]), 'positive': int(y[:, j].sum()),
                                 'negative': int((y[:, j] == 0).sum()),
                                 'perfect_target_max_macro_gain': float((1 - baseline_aucs[j]) / 12)}
    for stage in ('stage1', 'stage2', 'stage3', 'stage4'):
        frame = diagnostics.align(pd.read_csv(run.folder / (stage + '.csv'), dtype={GOLD_UID: str}), run.ids)
        scores = diagnostics.aucs(y, frame[GOLD_TARGETS].to_numpy(float))
        result['stage_macro_auc'][stage] = float(scores.mean())
    # Fixed candidates declared before seeing labels. No auto-selection/promotion.
    for name, spec in ablations['candidates'].items():
        if name == 'v13_control':
            continue
        candidate = pd.read_csv(spec['path'], dtype={GOLD_UID: str})
        try:
            comparison = diagnostics.compare(labels, baseline, candidate, bootstrap=2000, seed=1400)
        except ValueError as error:
            aligned = diagnostics.align(candidate, run.ids)
            scores = diagnostics.aucs(y, aligned[GOLD_TARGETS].to_numpy(float))
            comparison = {'candidate_macro_auc': float(scores.mean()),
                          'macro_delta': float((scores - baseline_aucs).mean()),
                          'bootstrap_unavailable_reason': str(error)}
        comparison['independent_confirmation'] = False
        result['candidates'][name] = comparison
        gold_write(run.work / (name + '_exploratory.json'), comparison)
    gold_write(run.work / 'gold_results.json', result)
    pd.DataFrame(result['baseline']).T.rename_axis('target').reset_index().to_csv(
        run.work / 'per_target_auc.csv', index=False)
    pd.DataFrame([{'candidate': name, 'macro_auc': item['candidate_macro_auc'],
                   'delta': item['macro_delta'], 'status': 'EXPLORATORY_ONLY'}
                  for name, item in result['candidates'].items()]).to_csv(
        run.work / 'candidate_summary.csv', index=False)
    print('EXPLORATORY V13 GOLD AUC:', result['macro_auc'], flush=True)
    print(pd.DataFrame(result['baseline']).T.to_string(), flush=True)
    print('EXPLORATORY CANDIDATE DELTAS:', json.dumps({k: v['macro_delta'] for k, v in result['candidates'].items()}), flush=True)
    print('NO LEADERBOARD GAIN OR +0.02 GUARANTEE IS ESTABLISHED.', flush=True)
    return result

_real = next((p for p in ([Path(os.environ['RSNA_DATA_ROOT'])] if os.environ.get('RSNA_DATA_ROOT') else []) + [
    Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
    Path('/kaggle/input/rsna-knee-abnormality-detection'),
    Path('data/rsna-knee-abnormality-detection'), Path('rsna-knee-abnormality-detection')] if (p / 'train.csv').is_file()), None)
assert _real is not None, 'Attach RSNA Knee competition data, or set RSNA_DATA_ROOT'
_ROOT, WORK = prepare_gold(_real, Path('/kaggle/working/family_a_gold_prep'), '/kaggle/temp')


In [ ]:
from __future__ import annotations
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_v, '4')
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

def _cuda_execution_probe(index):
    dev = torch.device(f'cuda:{index}')
    try:
        major, minor = torch.cuda.get_device_capability(index)
        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)
        with torch.inference_mode():
            out = probe(torch.zeros((1, 3, 16, 16), device=dev))
            if tuple(out.shape) != (1, 4, 16, 16):
                raise RuntimeError(f'unexpected CUDA probe shape {tuple(out.shape)}')
        torch.cuda.synchronize(index)
        print(f'cuda:{index} probe PASS (compute {major}.{minor})')
        del probe, out
        torch.cuda.empty_cache()
        return True
    except Exception as exc:
        print(f'cuda:{index} probe FAIL ({type(exc).__name__}: {exc}); using CPU fallback')
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False
DEVS = []
if torch.cuda.is_available():
    DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count()) if _cuda_execution_probe(i)]
if not DEVS:
    DEVS = [torch.device('cpu')]
print(f'devices: {[str(d) for d in DEVS]}')
T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
RUNS = [{'name': 'r224', 'img': 224}, {'name': 'r336', 'img': 336}]
EPOCHS = 10
BATCH_STUDIES = 8
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
LR_HEAD = 0.001
LR_BACKBONE = 8e-06
UNFREEZE_LAST = 6
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

In [ ]:
def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)

def find_root():
    return _ROOT
    for c in [Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'), Path('/kaggle/input/rsna-knee-abnormality-detection'), Path('data'), Path('.')]:
        if (c / 'test.csv').is_file() and (c / 'test_series').is_dir():
            return c
    base = Path('/kaggle/input')
    if base.is_dir():
        for depth1 in sorted((p for p in base.iterdir() if p.is_dir())):
            for cand in [depth1] + sorted((p for p in depth1.iterdir() if p.is_dir())):
                if (cand / 'test.csv').is_file():
                    return cand
    raise FileNotFoundError(f'competition mount not found (cwd {Path.cwd()}); expected a directory holding test.csv and test_series/')

def find_dinov2(variant='small'):
    base = Path('/kaggle/input')
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if 'config.json' in files and 'dinov2' in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None
LABEL_COLS = TARGETS + [t + '__conf' for t in TARGETS]

class LabelSourceError(RuntimeError):
    pass

def find_label_table():
    base = Path('/kaggle/input')
    cands = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
            cands += [Path(root) / f for f in files if f.startswith('report_labels') and f.endswith('.csv')]
    cands += [p for p in (Path('data/derived/report_labels_v2.csv'),) if p.is_file()]
    for c in cands:
        try:
            head = pd.read_csv(c, nrows=1)
        except Exception:
            continue
        if 'StudyInstanceUID' in head.columns and all((t in head.columns for t in TARGETS)):
            return c
    return None

def label_mount_attached():
    base = Path('/kaggle/input')
    if not base.is_dir():
        return False
    return any(('label' in p.name.lower() for p in base.iterdir() if p.is_dir()))

def read_labels(train_df):
    n = len(train_df)
    lab = pd.DataFrame([extract(r) for r in train_df['Report'].fillna('')])
    lab['StudyInstanceUID'] = train_df['StudyInstanceUID'].values
    lab = lab.set_index('StudyInstanceUID')
    src = find_label_table()
    if src is None:
        if label_mount_attached():
            raise LabelSourceError('LABEL SOURCE: a label dataset is mounted but no usable table was found in it. Falling back to the lexicon here would train on the weaker labels and say so only in a log line, so the run stops instead.')
        log(f'LABEL SOURCE: lexicon, {n} studies (no table mounted)')
        return lab
    tab = pd.read_csv(src).set_index('StudyInstanceUID')
    missing = [c for c in LABEL_COLS if c not in tab.columns]
    if missing:
        raise LabelSourceError(f'LABEL SOURCE: {src} is missing {len(missing)} expected columns (first: {missing[0]!r}). Refusing to fall back silently.')
    hit = lab.index.intersection(tab.index)
    if not len(hit):
        raise LabelSourceError(f'LABEL SOURCE: {src} shares no StudyInstanceUID with train.csv.')
    log(f'LABEL SOURCE: {src.name} covers {len(hit)} of {n} studies, lexicon for the remaining {n - len(hit)}')
    lab.loc[hit, LABEL_COLS] = tab.loc[hit, LABEL_COLS].values
    return lab
ROOT = find_root()
log(f'input root: {ROOT}')
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
log(f'cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot')

In [ ]:
HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']

def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split('|')]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None

def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)
        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)
        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))
        if ipp is None or iop is None or ps is None or (not rows) or (not cols):
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'
    return out

def side_from_corner_x(h):
    out = {}
    for st, g in h.groupby('StudyInstanceUID'):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'
    return out

def lat_of(h, tag=''):
    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)
    for st, g in h.groupby('StudyInstanceUID'):
        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]
        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:
            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]
        v = [x[0] for x in v if x and x[0] in ('L', 'R')]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')
    return d

def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)

def annotate(df):
    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))
    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')
    return df

In [ ]:
def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)
    out = {}
    for study, g in series_df.groupby('StudyInstanceUID'):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs)
            if fluid is not None:
                sel &= g['fluid'] == fluid
            cand = g[sel]
            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):
                cand = g[(g['plane'] == plane) & ~g['fatsat']]
            if len(cand):
                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]
        out[study] = chosen
    return out

In [ ]:
ORDER_TAGS = [(32, 50), (32, 55), (32, 19)]
DECODE_FAILED = []




def _natural_key(name):
    return tuple((int(x) if x.isdigit() else x.lower() for x in re.split('(\\d+)', str(name))))

def _order_dominant_axis(rec):
    files, d = (rec['files'], rec['dir'])
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            raw = getattr(ds, 'ImagePositionPatient', None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, 'InstanceNumber', None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))
    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float('inf'), r[3]))
    elif sum((r[2] is not None for r in rows)) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float('inf'), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return ([r[0] for r in rows], True)

def order_slices(rec):
    if RULES['order'] == 'dominant_axis':
        return _order_dominant_axis(rec)
    files, d = (rec['files'], rec['dir'])
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any((k is None for k, _ in keyed)):
        return (files, False)
    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)

def read_slot(rec, n_slice=None, out_size=None):
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = (rec.get('ordered') or rec['files'], rec['dir'], rec['px'])
    n = len(files)
    if n == 0:
        return None
    lo, hi = (int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1)))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])
    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, 'RescaleSlope', 1) or 1)
            ic = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None
        planes.append(a)
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES['decode_fill'] == 'zero':
        if not got:
            DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)
    if px and np.isfinite(px) and (px > 0):
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = (h // 2, w // 2)
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-06), 0, 1)
    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode='bilinear', align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


In [ ]:
import sys
from pathlib import Path
_src = Path('/kaggle/working/family_a_src')
_src.mkdir(parents=True, exist_ok=True)
(_src / 'contract.py').write_text('"""Tensor/metadata contract shared by every Family A component.\n\nMirrors the already-deployed, already-tested V13 six-slot layout (SLOTS, GROUP,\nIMG, CROP_MM, SLOT_PRIOR_TABLE) so a raw-slot-tensor cache built by reusing V13\'s\nown pick_slots/order_slices/read_slot cells (see build_pool_notebook.py) is\ndirectly consumable here without a second, independently-written and unvalidated\nDICOM decoder. Family A\'s model and supervision are new; the pixel pipeline is\nreused, not reinvented.\n"""\n\nUID = \'StudyInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n\n# name, plane, fluid-sensitive (None = don\'t care), fatsat\nSLOTS = [\n    (\'SAG_FLUID_FS\', \'Sagittal\', True, True),\n    (\'COR_FLUID_FS\', \'Coronal\', True, True),\n    (\'AX_FLUID_FS\', \'Axial\', True, True),\n    (\'SAG_FLUID_NOFS\', \'Sagittal\', True, False),\n    (\'COR_T1\', \'Coronal\', False, False),\n    (\'SAG_T1\', \'Sagittal\', False, False),\n]\nN_SLOT = len(SLOTS)\nGROUP = 3        # adjacent slices per window, fed as pseudo-RGB channels\nIMG = 336        # per-slot tile side, pixels\nCROP_MM = 130.0\n\nSLOT_PRIOR_TABLE = {\n    \'ACL\': (0, 3, 5), \'MCL\': (1, 4), \'Medial Meniscus\': (0, 1, 3, 4),\n    \'Lateral Meniscus\': (0, 1, 3, 4), \'Medial OA\': (1, 4, 5), \'Lateral OA\': (1, 4, 5),\n    \'PF OA\': (0, 2, 5), \'Effusion\': (0, 2), \'Synovitis\': (0, 2), "Baker\'s": (0,),\n    \'Contusion\': (0, 1, 2), \'Fracture\': (0, 1, 2, 4, 5),\n}\nSLOT_PRIOR_STRENGTH = 0.55\n', encoding='utf-8')
(_src / 'labels.py').write_text('"""Label construction: expert gold labels are the primary objective; a capped,\nevidence-gated auxiliary target from report-derived soft labels is optional.\n\nCorrections applied per review, encoded here rather than left as prose policy:\n\n  * Report-silent cells (raw value == 0.5) are EXCLUDED from the auxiliary loss\n    (mask weight 0), never filled in as a hard negative. "Masked" and "filled as\n    negative" are different treatments; Run 4 uses the former exclusively.\n  * A target gets nonzero auxiliary weight only if its transfer_audit.py\n    addressed-only AUC bootstrap CI lower bound exceeds 0.5 AND its Brier skill\n    score (vs. the prevalence-only baseline) is positive. AUC alone is not\n    sufficient: MCL has AUC 0.958 but skill only 0.11 (near-chance calibration\n    despite near-perfect ranking); Lateral OA/Synovitis/Contusion have AUC CI\n    lower bounds above 0.5 but NEGATIVE skill and are excluded despite passing\n    on AUC alone.\n  * A target flagged small_sample_warning (fewer than 10 studies in the smaller\n    addressed class) has its weight halved rather than trusted at face value.\n  * This policy is derived from the same 58 gold studies Run 4 later scores\n    against. Any Run 4 result on that same 58-study cohort is exploratory, not\n    independent confirmation of the policy that selected it.\n"""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom contract import TARGETS, UID\n\nMAX_AUX_WEIGHT = 0.3\nSMALL_SAMPLE_WEIGHT_MULTIPLIER = 0.5\nSILENT_VALUE = 0.5\n\n\ndef load_transfer_policy(transfer_audit_report_path):\n    """Derive {target: {\'weight\': float, \'use_aux\': bool, \'reason\': str}} from a\n    transfer_audit.py report. Every decision carries its numeric justification\n    so the policy is auditable, not a hand-picked list."""\n    report = json.loads(Path(transfer_audit_report_path).read_text(encoding=\'utf-8\'))\n    return policy_from_audit(report)\n\n\ndef policy_from_audit(report):\n    """Convert an in-memory transfer audit to the capped auxiliary policy."""\n    policy = {}\n    for t in TARGETS:\n        d = report[\'targets\'][t]\n        auc_ci = d[\'auc_addressed_only\'][\'ci95\']\n        skill = d[\'brier_addressed_only\'][\'skill_score\']\n        if auc_ci is None or skill is None:\n            policy[t] = {\'weight\': 0.0, \'use_aux\': False,\n                         \'reason\': \'insufficient evidence: AUC CI or Brier skill unavailable at this n\'}\n            continue\n        ci_lower = auc_ci[0]\n        if ci_lower <= 0.5 or skill <= 0.0:\n            policy[t] = {\'weight\': 0.0, \'use_aux\': False,\n                         \'reason\': f\'gate failed: AUC CI lower={ci_lower:.3f}, skill={skill:.3f}\'}\n            continue\n        weight = min(skill, MAX_AUX_WEIGHT)\n        halved = bool(d[\'small_sample_warning\'])\n        if halved:\n            weight *= SMALL_SAMPLE_WEIGHT_MULTIPLIER\n        policy[t] = {\'weight\': float(weight), \'use_aux\': True,\n                     \'reason\': f\'gate passed: AUC CI lower={ci_lower:.3f}, skill={skill:.3f}\'\n                               + (\', weight halved (small sample)\' if halved else \'\')}\n    return policy\n\n\ndef crossfit_transfer_policy(ids, train_idx, gold_labels, report_source,\n                             bootstrap=1000, seed=1400):\n    """Derive a policy using training-fold expert cases only.\n\n    This closes the policy-selection leak that occurs when a policy selected on\n    all 58 gold cases is evaluated on OOF predictions for those same cases.\n    """\n    try:\n        from transfer_audit import audit_target\n    except ModuleNotFoundError:  # repository-root test invocation\n        from v15.transfer_audit import audit_target\n\n    ids = np.asarray(ids, dtype=str)\n    train_ids = set(ids[np.asarray(train_idx, dtype=int)])\n    gold = gold_labels.set_index(UID) if UID in gold_labels.columns else gold_labels\n    report = report_source.set_index(UID) if UID in report_source.columns else report_source\n    if gold.index.duplicated().any() or report.index.duplicated().any():\n        raise ValueError(\'Duplicate study IDs cannot be used for cross-fitted policy selection\')\n    selected = [uid for uid in gold.index.astype(str) if uid in train_ids]\n    if not selected:\n        raise ValueError(\'No training-fold expert cases available for policy selection\')\n    if not set(selected).issubset(set(report.index.astype(str))):\n        raise ValueError(\'Training-fold expert cases are missing from report source\')\n    y = gold.loc[selected, TARGETS].to_numpy(float)\n    p = report.loc[selected, TARGETS].to_numpy(float)\n    if not np.isin(y, [0.0, 1.0]).all():\n        raise ValueError(\'Cross-fitted policy requires fully binary expert labels\')\n    targets = {t: audit_target(y[:, j], p[:, j], bootstrap=bootstrap,\n                               seed=seed + j)\n               for j, t in enumerate(TARGETS)}\n    audit = {\'cohort_studies\': len(selected), \'bootstrap\': bootstrap,\n             \'seed\': seed, \'targets\': targets}\n    return policy_from_audit(audit), audit\n\n\ndef zero_policy():\n    """The baseline-arm policy: auxiliary loss disabled for every target."""\n    return {t: {\'weight\': 0.0, \'use_aux\': False, \'reason\': \'baseline arm: auxiliary loss disabled\'}\n            for t in TARGETS}\n\n\ndef build_supervision(ids, gold_labels, report_source, policy, silent_value=SILENT_VALUE):\n    """Return (y_expert, expert_mask, y_aux, aux_mask, aux_weight) aligned to `ids`.\n\n    y_expert/expert_mask: expert gold labels where the study is in the gold\n    cohort, NaN/False elsewhere. This is the only signal Run 4 evaluates against.\n\n    y_aux/aux_mask: report-derived soft value, masked out (never filled as a\n    negative) wherever the report was silent OR the target failed the transfer\n    policy gate. Fold-safety (excluding validation-fold rows from any loss) is\n    the training loop\'s responsibility, not this function\'s -- these arrays are\n    fold-agnostic raw supervision.\n\n    aux_weight: one capped scalar per target from the policy, constant across\n    studies.\n    """\n    ids = list(ids)\n    n = len(ids)\n    y_expert = np.full((n, len(TARGETS)), np.nan)\n    expert_mask = np.zeros((n, len(TARGETS)), dtype=bool)\n    if gold_labels is not None and len(gold_labels):\n        gold = gold_labels.set_index(UID) if UID in gold_labels.columns else gold_labels\n        if gold.index.duplicated().any():\n            raise ValueError(\'Duplicate study IDs in gold labels\')\n        for i, uid in enumerate(ids):\n            if uid in gold.index:\n                row = gold.loc[uid, TARGETS].to_numpy(float)\n                y_expert[i] = row\n                expert_mask[i] = np.isfinite(row)\n\n    report = report_source.set_index(UID) if UID in report_source.columns else report_source\n    if report.index.duplicated().any():\n        raise ValueError(\'Duplicate study IDs in report source\')\n    missing = [u for u in ids if u not in report.index]\n    if missing:\n        raise ValueError(f\'{len(missing)} studies missing from report source, e.g. {missing[:3]}\')\n    raw = report.loc[ids, TARGETS].to_numpy(float)\n    if not (np.isfinite(raw) & (raw >= 0) & (raw <= 1)).all():\n        raise ValueError(\'Report-derived values must be finite probabilities in [0, 1]\')\n\n    addressed = raw != silent_value\n    use_aux = np.array([policy.get(t, {}).get(\'use_aux\', False) for t in TARGETS])\n    # Expert truth takes precedence. Report supervision is used only for cells\n    # that do not already carry an expert label, avoiding contradictory double\n    # supervision on the 58 gold studies.\n    aux_mask = addressed & use_aux[None, :] & ~expert_mask\n    y_aux = np.where(aux_mask, raw, np.nan)\n    aux_weight = np.array([policy.get(t, {}).get(\'weight\', 0.0) for t in TARGETS], dtype=float)\n    return y_expert, expert_mask, y_aux, aux_mask, aux_weight\n', encoding='utf-8')
(_src / 'model.py').write_text('"""Family A: six-slot 2.5D multiple-instance model with target-query attention.\n\nThe pooling head\'s math matches the deployed V13 SlotHead exactly (proj -> add\nslot embedding -> per-target query attention over slots, masked-softmax, output\nprojection, optional slot-anatomy prior bias) since that mechanism is already\nvalidated in production; what\'s new is the encoder is trainable here (unfrozen\nlast blocks, per the plan\'s Phase A1) rather than used only as a frozen feature\nsource the way the failed specialist experiments did, and the model is trained\ndirectly (with capped auxiliary supervision), not as a post-hoc residual\ncorrection on top of another model\'s output.\n\nEncoders are pluggable so architecture correctness can be verified on CPU with\nTinyCNNEncoder (no external weights, no GPU) before ever touching a real image\nor the transformers/DINOv2 dependency, which Dinov2Encoder needs and which only\nmatters on the actual training device.\n"""\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom contract import GROUP, IMG, N_SLOT, SLOT_PRIOR_STRENGTH, SLOT_PRIOR_TABLE, SLOTS, TARGETS\n\n\nclass TargetQueryHead(nn.Module):\n    """One learned query per target attends over the N_SLOT slot tokens."""\n\n    def __init__(self, dim, n_slot=N_SLOT, n_out=len(TARGETS), hidden=256, dropout=0.2, use_prior=True):\n        super().__init__()\n        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())\n        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)\n        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(hidden, n_out)\n        self.hidden = hidden\n        self.use_prior = use_prior and n_slot == len(SLOTS) and n_out == len(TARGETS)\n        prior = torch.zeros(n_out, n_slot)\n        if self.use_prior:\n            for t, slots in SLOT_PRIOR_TABLE.items():\n                if t in TARGETS:\n                    prior[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH\n        self.register_buffer(\'slot_prior\', prior)\n\n    def forward(self, tokens, mask):\n        """tokens: (B, n_slot, dim). mask: (B, n_slot) in {0, 1}, 1 = slot present.\n        Returns (B, n_out) logits and (B, n_out, n_slot) attention weights."""\n        h = self.proj(tokens) + self.slot_emb\n        att = torch.einsum(\'bsh,oh->bos\', h, self.query) / self.hidden ** 0.5\n        if self.use_prior:\n            att = att + self.slot_prior.unsqueeze(0)\n        if (mask.sum(dim=1) == 0).any():\n            raise ValueError(\'At least one slot must be present per study; an all-missing study cannot be scored\')\n        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)\n        ctx = self.drop(torch.einsum(\'bos,bsh->boh\', att, h))\n        logits = (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias\n        return logits, att.detach()\n\n\nclass TinyCNNEncoder(nn.Module):\n    """CPU-only stand-in encoder for architecture/shape/gradient-flow testing.\n    Not a modeling contribution -- swap in Dinov2Encoder for real training."""\n\n    def __init__(self, out_dim=64):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(GROUP, 16, 5, stride=4, padding=2), nn.GELU(),\n            nn.Conv2d(16, 32, 5, stride=4, padding=2), nn.GELU(),\n            nn.AdaptiveAvgPool2d(1),\n        )\n        self.proj = nn.Linear(32, out_dim)\n        self.out_dim = out_dim\n\n    def forward(self, x):\n        """x: (N, GROUP, IMG, IMG) float in [0, 1]. Returns (N, out_dim)."""\n        return self.proj(self.net(x).flatten(1))\n\n\nclass Dinov2Encoder(nn.Module):\n    """Wraps a HuggingFace DINOv2 backbone with partial unfreezing, matching the\n    already-tested V13 build_model() unfreeze/normalize contract exactly, so a\n    checkpoint trained here is not fighting a second, different preprocessing\n    convention. Only imported/instantiated on the GPU training device; the\n    `transformers` dependency and pretrained weights are not needed for the CPU\n    smoke test."""\n\n    def __init__(self, source_path, unfreeze_last=2, pool=\'cls_mean\'):\n        super().__init__()\n        from transformers import AutoModel\n        backbone = AutoModel.from_pretrained(str(source_path))\n        n_layer = len(backbone.encoder.layer)\n        for p in backbone.parameters():\n            p.requires_grad = False\n        for blk in backbone.encoder.layer[max(0, n_layer - unfreeze_last):]:\n            for p in blk.parameters():\n                p.requires_grad = True\n        for p in backbone.layernorm.parameters():\n            p.requires_grad = True\n        self.backbone = backbone\n        self.pool = pool\n        parts = {\'cls_mean\': 2, \'cls_mean_focal\': 3}[pool]\n        self.out_dim = backbone.config.hidden_size * parts\n        self.register_buffer(\'mean\', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))\n        self.register_buffer(\'std\', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))\n\n    def forward(self, x):\n        x = (x - self.mean) / self.std\n        out = self.backbone(pixel_values=x).last_hidden_state\n        patch = out[:, 1:]\n        parts = [out[:, 0], patch.mean(1)]\n        if self.pool == \'cls_mean_focal\':\n            k = max(1, patch.shape[1] // 8)\n            parts.append(patch.topk(k, dim=1).values.mean(1))\n        return torch.cat(parts, dim=1)\n\n\nclass GeneralistModel(nn.Module):\n    def __init__(self, encoder, n_slot=N_SLOT, n_out=len(TARGETS), head_hidden=256, dropout=0.2, use_prior=True):\n        super().__init__()\n        self.encoder = encoder\n        self.head = TargetQueryHead(encoder.out_dim, n_slot, n_out, head_hidden, dropout, use_prior)\n\n    def forward(self, imgs, mask, img_size=None):\n        """imgs: (B, n_slot, GROUP, H, W) uint8 or float. mask: (B, n_slot)."""\n        b, s = imgs.shape[:2]\n        x = imgs.reshape(b * s, *imgs.shape[2:]).float()\n        if x.max() > 1.5:  # tensor arrived as 0-255 uint8-range values\n            x = x / 255.0\n        if img_size is not None and img_size != x.shape[-1]:\n            x = F.interpolate(x, size=(img_size, img_size), mode=\'bilinear\', align_corners=False)\n        tokens = self.encoder(x).reshape(b, s, -1)\n        return self.head(tokens, mask)\n\n\ndef build_smoke_model(hidden=32, out_dim=16):\n    return GeneralistModel(TinyCNNEncoder(out_dim=out_dim), head_hidden=hidden)\n', encoding='utf-8')
(_src / 'folds.py').write_text('"""Deterministic grouped fold assignment.\n\nThe repository\'s own PatientID audit found singleton groups in the available\nmetadata (no verified repeat-patient linkage), so grouping by patient identity\ncannot currently be claimed here. This groups by StudyInstanceUID -- i.e. every\nstudy is its own group -- which is a weaker guarantee than true patient\nseparation and is labeled as such in every receipt this module contributes to,\nrather than silently presented as patient-safe splitting.\n"""\nimport numpy as np\n\n\ndef assign_folds(ids, k=5, seed=1400, priority_ids=None):\n    if k < 2:\n        raise ValueError(\'Need at least 2 folds\')\n    ids = list(ids)\n    n = len(ids)\n    if n < k:\n        raise ValueError(f\'Need at least as many studies ({n}) as folds ({k})\')\n    if len(set(ids)) != n:\n        raise ValueError(\'Duplicate study IDs cannot be assigned to folds\')\n    priority_ids = set() if priority_ids is None else set(priority_ids)\n    unknown = priority_ids - set(ids)\n    if unknown:\n        raise ValueError(f\'Priority set contains {len(unknown)} unknown study IDs\')\n    rng = np.random.default_rng(seed)\n    priority = np.array([i for i, uid in enumerate(ids) if uid in priority_ids], dtype=int)\n    ordinary = np.array([i for i, uid in enumerate(ids) if uid not in priority_ids], dtype=int)\n    fold_of = np.full(n, -1, dtype=int)\n    fold_order = rng.permutation(k)\n    for j, row in enumerate(rng.permutation(priority)):\n        fold_of[row] = fold_order[j % k]\n    counts = np.bincount(fold_of[fold_of >= 0], minlength=k)\n    tie_order = {fold: rank for rank, fold in enumerate(fold_order)}\n    for row in rng.permutation(ordinary):\n        fold = min(range(k), key=lambda f: (counts[f], tie_order[f]))\n        fold_of[row] = fold\n        counts[fold] += 1\n    return {\n        \'fold_assignment\': dict(zip(ids, fold_of.tolist())),\n        \'k\': k, \'seed\': seed, \'n_studies\': n,\n        \'priority_studies_balanced\': len(priority_ids),\n        \'grouping_caveat\': (\'Grouped by StudyInstanceUID, one study per group. Patient-level \'\n                            \'grouping was not used: the repository\\\'s metadata audit found only \'\n                            \'singleton patient groups, so patient-safe separation is unverified.\'),\n    }\n', encoding='utf-8')
(_src / 'losses.py').write_text('"""Masked BCE for expert labels, plus an optional capped, masked auxiliary term.\n\nBoth losses are computed with logits (not probabilities) via\nbinary_cross_entropy_with_logits for numerical stability, and both are averaged\nonly over the cells their own mask marks valid -- a study/target cell that is\nNaN in y (no expert label, or report-silent for the auxiliary target) never\nenters either loss, it is not treated as a zero.\n"""\nimport torch\nimport torch.nn.functional as F\n\n\ndef masked_bce(logits, y, mask):\n    """logits, y, mask: (B, T). Returns a scalar; 0.0 (not NaN) if mask is empty\n    so a batch with no expert-labeled rows for a given fold split doesn\'t produce\n    a NaN gradient step."""\n    if mask.sum() == 0:\n        return logits.new_zeros(())\n    y_safe = torch.where(mask, y, torch.zeros_like(y))\n    per_cell = F.binary_cross_entropy_with_logits(logits, y_safe, reduction=\'none\')\n    return (per_cell * mask.float()).sum() / mask.float().sum()\n\n\ndef combined_loss(logits, y_expert, expert_mask, y_aux, aux_mask, aux_weight):\n    """aux_weight: (T,) capped per-target scalar, broadcast across the batch and\n    multiplied into the per-cell auxiliary loss. The denominator is the number\n    of valid cells, not the sum of weights; otherwise uniformly shrinking every\n    weight would cancel out and the nominal cap would not cap the loss."""\n    expert_term = masked_bce(logits, y_expert, expert_mask)\n    if aux_weight is None or float(aux_weight.abs().sum()) == 0.0:\n        return expert_term, {\'expert\': float(expert_term.detach()), \'aux\': 0.0}\n    y_safe = torch.where(aux_mask, y_aux, torch.zeros_like(y_aux))\n    per_cell = F.binary_cross_entropy_with_logits(logits, y_safe, reduction=\'none\')\n    weighted_mask = aux_mask.float() * aux_weight.unsqueeze(0)\n    denom = aux_mask.float().sum()\n    aux_term = (per_cell * weighted_mask).sum() / denom if denom > 0 else logits.new_zeros(())\n    total = expert_term + aux_term\n    return total, {\'expert\': float(expert_term.detach()), \'aux\': float(aux_term.detach())}\n', encoding='utf-8')
(_src / 'cache.py').write_text('"""Chunked, checksum-backed slot-tensor cache: one file per study, never one\ngiant fragile array, per the plan\'s cache design. Real cache contents are built\non the GPU device by build_pool_notebook.py (which reuses V13\'s own\npick_slots/order_slices/read_slot cells against real DICOMs); synth() here\nbuilds a structurally identical fake cache from random tensors so every other\nFamily A component can be exercised on CPU without any competition data.\n"""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom contract import GROUP, IMG, N_SLOT, UID\n\n\ndef _study_path(cache_dir, uid):\n    if \'/\' in uid or \'\\\\\' in uid or uid in (\'.\', \'..\'):\n        raise ValueError(\'Unsafe study identifier\')\n    return Path(cache_dir) / f\'{uid}.npz\'\n\n\ndef write_study(cache_dir, uid, imgs, mask):\n    imgs = np.asarray(imgs)\n    mask = np.asarray(mask)\n    if imgs.shape != (N_SLOT, GROUP, IMG, IMG):\n        raise ValueError(f\'imgs must be shape {(N_SLOT, GROUP, IMG, IMG)}, got {imgs.shape}\')\n    if mask.shape != (N_SLOT,):\n        raise ValueError(f\'mask must be shape {(N_SLOT,)}, got {mask.shape}\')\n    if not mask.any():\n        raise ValueError(f\'{uid}: at least one slot must be present\')\n    if imgs.dtype != np.uint8:\n        raise ValueError(\'imgs must be uint8 (percentile-normalized, 0-255)\')\n    path = _study_path(cache_dir, uid)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(path, imgs=imgs, mask=mask.astype(np.float32))\n    return path\n\n\ndef build_manifest(cache_dir, ids):\n    cache_dir = Path(cache_dir)\n    entries = {}\n    missing = []\n    for uid in ids:\n        path = _study_path(cache_dir, uid)\n        if not path.is_file():\n            missing.append(uid)\n            continue\n        with np.load(path) as f:\n            imgs, mask = f[\'imgs\'], f[\'mask\']\n        if imgs.shape != (N_SLOT, GROUP, IMG, IMG) or mask.shape != (N_SLOT,):\n            raise ValueError(f\'{uid}: cached tensor does not match the current contract shape\')\n        entries[uid] = {\'path\': path.name, \'sha256\': hashlib.sha256(path.read_bytes()).hexdigest(),\n                        \'n_slots_present\': int(mask.sum())}\n    if missing:\n        raise FileNotFoundError(f\'{len(missing)} studies missing from cache, e.g. {missing[:3]}\')\n    manifest = {\'contract\': {\'n_slot\': N_SLOT, \'group\': GROUP, \'img\': IMG},\n                \'studies\': entries, \'n_studies\': len(entries)}\n    (cache_dir / \'manifest.json\').write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    return manifest\n\n\ndef load_manifest(cache_dir):\n    return json.loads((Path(cache_dir) / \'manifest.json\').read_text(encoding=\'utf-8\'))\n\n\ndef load_study(cache_dir, uid):\n    with np.load(_study_path(cache_dir, uid)) as f:\n        return f[\'imgs\'], f[\'mask\']\n\n\ndef load_batch(cache_dir, ids):\n    imgs, masks = [], []\n    for uid in ids:\n        i, m = load_study(cache_dir, uid)\n        imgs.append(i)\n        masks.append(m)\n    return np.stack(imgs), np.stack(masks)\n\n\ndef synth(cache_dir, ids, seed=1400, min_slots_present=3):\n    """Build a structurally valid fake cache for the CPU smoke test. Not real\n    image data; exercises every downstream shape/mask/manifest code path."""\n    rng = np.random.default_rng(seed)\n    for uid in ids:\n        n_present = rng.integers(min_slots_present, N_SLOT + 1)\n        mask = np.zeros(N_SLOT, dtype=bool)\n        mask[rng.choice(N_SLOT, size=n_present, replace=False)] = True\n        imgs = rng.integers(0, 256, size=(N_SLOT, GROUP, IMG, IMG), dtype=np.uint8)\n        write_study(cache_dir, uid, imgs, mask)\n    return build_manifest(cache_dir, ids)\n', encoding='utf-8')
(_src / 'engine.py').write_text('"""Training loop: deterministic seeding, grouped folds, checkpoint resume,\nper-arm (baseline / auxiliary) supervision, OOF export, and a receipt for every\nrun. Loss is computed on the training split only; validation-fold rows are\nscored with a bare forward pass (no grad) purely to produce the OOF prediction\nfor that row -- this is also why no special-casing is needed to keep a fold\'s\nheld-out expert label from leaking through the auxiliary term: the auxiliary\nloss is only ever summed over training-split rows in the first place.\n"""\nimport hashlib\nimport json\nimport os\nimport random\nimport subprocess\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom compare_oof import aucs\nfrom contract import TARGETS, UID\nfrom folds import assign_folds\nfrom labels import build_supervision, crossfit_transfer_policy\nfrom losses import combined_loss\n\n\ndef resolve_device(requested=\'auto\'):\n    """Return a usable device after a real CUDA arithmetic probe."""\n    if requested not in (\'auto\', \'cpu\', \'cuda\'):\n        raise ValueError("device must be \'auto\', \'cpu\', or \'cuda\'")\n    if requested == \'cpu\':\n        return torch.device(\'cpu\')\n    try:\n        if torch.cuda.is_available():\n            probe = torch.ones(1, device=\'cuda\') * 2\n            if float(probe.cpu()) != 2.0:\n                raise RuntimeError(\'CUDA arithmetic probe returned the wrong value\')\n            torch.cuda.synchronize()\n            return torch.device(\'cuda\')\n    except Exception as exc:\n        if requested == \'cuda\':\n            raise RuntimeError(f\'CUDA failed a real arithmetic probe: {exc}\') from exc\n    if requested == \'cuda\':\n        raise RuntimeError(\'CUDA was required but is unavailable\')\n    return torch.device(\'cpu\')\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    if hasattr(torch.backends, \'cudnn\'):\n        torch.backends.cudnn.benchmark = False\n\n\ndef git_commit():\n    try:\n        return subprocess.run([\'git\', \'rev-parse\', \'HEAD\'], capture_output=True, text=True,\n                              cwd=Path(__file__).resolve().parent, check=True).stdout.strip()\n    except Exception:\n        return None\n\n\ndef digest_file(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef atomic_save(path, state):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    torch.save(state, tmp)\n    os.replace(tmp, path)\n\n\ndef write_receipt(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(data, f, indent=2, default=str)\n        f.write(\'\\n\')\n\n\ndef rng_state(batch_rng=None):\n    state = {\'python\': random.getstate(), \'numpy\': np.random.get_state(),\n             \'torch\': torch.get_rng_state()}\n    try:\n        if torch.cuda.is_initialized():\n            state[\'torch_cuda\'] = torch.cuda.get_rng_state_all()\n    except Exception:\n        pass\n    if batch_rng is not None:\n        state[\'batch_rng\'] = batch_rng.bit_generator.state\n    return state\n\n\ndef restore_rng_state(state):\n    random.setstate(state[\'python\'])\n    np.random.set_state(state[\'numpy\'])\n    torch.set_rng_state(state[\'torch\'])\n    if \'torch_cuda\' in state and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all(state[\'torch_cuda\'])\n\n\ndef _batches(idx, batch_size, rng):\n    idx = idx.copy()\n    rng.shuffle(idx)\n    for start in range(0, len(idx), batch_size):\n        yield idx[start:start + batch_size]\n\n\ndef _epoch_training_indices(train_idx, expert_mask, aux_mask, aux_weight,\n                            expert_fraction, rng):\n    """Visit all auxiliary-only rows and sample enough expert rows to anchor them.\n\n    With 46 training-side gold studies among roughly 3,500 report-only studies,\n    a plain shuffle exposes the expert objective in only ~1.3% of rows. The\n    declared fraction prevents that accidental dilution without discarding any\n    eligible auxiliary row.\n    """\n    train_idx = np.asarray(train_idx, dtype=int)\n    weighted_aux = aux_mask & (aux_weight[None, :] > 0)\n    expert_rows = train_idx[expert_mask[train_idx].any(axis=1)]\n    aux_only_rows = train_idx[(~expert_mask[train_idx].any(axis=1)) &\n                              weighted_aux[train_idx].any(axis=1)]\n    if not len(expert_rows):\n        return aux_only_rows\n    if not len(aux_only_rows):\n        return expert_rows\n    if not 0 < expert_fraction < 1:\n        raise ValueError(\'expert_fraction must be strictly between 0 and 1\')\n    requested = max(len(expert_rows), int(np.ceil(\n        expert_fraction * len(aux_only_rows) / (1 - expert_fraction))))\n    sampled_expert = rng.choice(expert_rows, size=requested,\n                                replace=requested > len(expert_rows))\n    return np.concatenate([aux_only_rows, sampled_expert])\n\n\ndef _predict(model, imgs_t, masks_t, idx, device, batch_size=8, use_amp=False):\n    model.eval()\n    probabilities, attentions = [], []\n    idx = np.asarray(idx)\n    with torch.no_grad():\n        for start in range(0, len(idx), batch_size):\n            batch = idx[start:start + batch_size]\n            x = imgs_t[batch].to(device, non_blocking=True)\n            m = masks_t[batch].to(device, non_blocking=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):\n                logits, attention = model(x, m)\n            probabilities.append(torch.sigmoid(logits).float().cpu().numpy())\n            attentions.append(attention.float().cpu().numpy())\n    return np.concatenate(probabilities), np.concatenate(attentions)\n\n\ndef _val_macro_auc(probs, y_expert, expert_mask, val_idx):\n    """Macro AUC over whichever val-fold rows actually carry an expert label\n    (report-only pool studies in the validation split contribute nothing here,\n    same as they contribute nothing to the final scored OOF)."""\n    val_idx = np.array(val_idx)\n    gold_rows = expert_mask[val_idx].any(axis=1)\n    if gold_rows.sum() < 4:\n        return None  # too few gold rows in this fold\'s validation split to trust a per-epoch AUC\n    y = np.where(expert_mask[val_idx][gold_rows], y_expert[val_idx][gold_rows], np.nan)\n    per_target = aucs(y, probs[gold_rows])\n    valid = per_target[np.isfinite(per_target)]\n    return float(valid.mean()) if len(valid) else None\n\n\ndef build_optimizer(model, head_lr=1e-3, backbone_lr=8e-6, weight_decay=0.02):\n    """AdamW with a conservative rate for the partially unfrozen encoder.\n\n    The target-query head is initialized from scratch, whereas the encoder is\n    pretrained and only its final blocks are trainable. Updating both at the\n    head rate destroys the representation Family A is intended to transfer.\n    """\n    encoder_ids = {id(p) for p in getattr(model, \'encoder\', ()).parameters()} \\\n        if hasattr(model, \'encoder\') else set()\n    encoder = [p for p in model.parameters() if p.requires_grad and id(p) in encoder_ids]\n    head = [p for p in model.parameters() if p.requires_grad and id(p) not in encoder_ids]\n    groups = []\n    if head:\n        groups.append({\'params\': head, \'lr\': head_lr, \'group_name\': \'head\'})\n    if encoder:\n        groups.append({\'params\': encoder, \'lr\': backbone_lr, \'group_name\': \'encoder\'})\n    if not groups:\n        raise ValueError(\'Model has no trainable parameters\')\n    return torch.optim.AdamW(groups, weight_decay=weight_decay)\n\n\ndef train_one_fold(model_fn, imgs, masks, y_expert, expert_mask, y_aux, aux_mask, aux_weight,\n                    train_idx, val_idx, epochs, lr, seed, checkpoint_path, batch_size=8, resume=True,\n                    patience=None, device=\'auto\', amp=None, prediction_batch_size=None,\n                    backbone_lr=8e-6, weight_decay=0.02, expert_fraction=0.10):\n    """Train a fixed, predeclared epoch count and score the outer fold once.\n\n    Outer-fold expert labels are not inspected during training or used for\n    checkpoint selection. `patience` is rejected because early stopping on the\n    outer fold would make the resulting OOF estimate optimistic.\n    """\n    if epochs < 1:\n        raise ValueError(\'epochs must be at least 1\')\n    if patience is not None:\n        raise ValueError(\'Outer-fold early stopping is prohibited; use fixed epochs\')\n    device = resolve_device(device)\n    use_amp = device.type == \'cuda\' if amp is None else bool(amp and device.type == \'cuda\')\n    prediction_batch_size = prediction_batch_size or batch_size\n    set_seed(seed)\n    model = model_fn().to(device)\n    optimizer = build_optimizer(model, head_lr=lr, backbone_lr=backbone_lr,\n                                weight_decay=weight_decay)\n    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)\n    training_config = {\'head_lr\': lr, \'backbone_lr\': backbone_lr,\n                       \'weight_decay\': weight_decay, \'batch_size\': batch_size,\n                       \'expert_fraction\': expert_fraction}\n    start_epoch = 0\n    stopping_reason = \'completed_predeclared_fixed_epochs\'\n    checkpoint_path = Path(checkpoint_path)\n    history = []\n    batch_rng = np.random.default_rng(seed + 1)\n    if resume and checkpoint_path.is_file():\n        state = torch.load(checkpoint_path, map_location=device, weights_only=False)\n        if state.get(\'training_config\') != training_config:\n            raise ValueError(\'Checkpoint training configuration differs from this run; \'\n                             \'use a fresh output directory\')\n        model.load_state_dict(state[\'model\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        if \'scaler\' in state:\n            scaler.load_state_dict(state[\'scaler\'])\n        restore_rng_state(state[\'rng_state\'])\n        start_epoch = state[\'epoch\'] + 1\n        history = state.get(\'history\', [])\n        if \'batch_rng\' in state[\'rng_state\']:\n            batch_rng.bit_generator.state = state[\'rng_state\'][\'batch_rng\']\n\n    imgs_t = torch.from_numpy(imgs)\n    masks_t = torch.from_numpy(masks).float()\n    y_expert_t = torch.from_numpy(np.nan_to_num(y_expert, nan=0.0)).float()\n    expert_mask_t = torch.from_numpy(expert_mask)\n    y_aux_t = torch.from_numpy(np.nan_to_num(y_aux, nan=0.0)).float()\n    aux_mask_t = torch.from_numpy(aux_mask)\n    aux_weight_t = torch.from_numpy(aux_weight).float()\n\n    train_idx = np.asarray(train_idx, dtype=int)\n    if not (expert_mask[train_idx].any() or\n            (aux_mask[train_idx] & (aux_weight[None, :] > 0)).any()):\n        raise ValueError(\'No supervised rows remain in this training fold\')\n\n    for epoch in range(start_epoch, epochs):\n        model.train()\n        epoch_losses = []\n        epoch_idx = _epoch_training_indices(\n            train_idx, expert_mask, aux_mask, aux_weight, expert_fraction, batch_rng)\n        expert_instances = int(expert_mask[epoch_idx].any(axis=1).sum())\n        for batch in _batches(epoch_idx, batch_size, batch_rng):\n            optimizer.zero_grad()\n            x = imgs_t[batch].to(device, non_blocking=True)\n            m = masks_t[batch].to(device, non_blocking=True)\n            ye = y_expert_t[batch].to(device, non_blocking=True)\n            em = expert_mask_t[batch].to(device, non_blocking=True)\n            ya = y_aux_t[batch].to(device, non_blocking=True)\n            am = aux_mask_t[batch].to(device, non_blocking=True)\n            aw = aux_weight_t.to(device, non_blocking=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):\n                logits, _ = model(x, m)\n                loss, parts = combined_loss(logits, ye, em, ya, am, aw)\n            if not torch.isfinite(loss):\n                raise RuntimeError(f\'non-finite loss at epoch {epoch}: {parts}\')\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            epoch_losses.append(parts)\n\n        history.append({\'epoch\': epoch,\n                        \'mean_expert_loss\': float(np.mean([p[\'expert\'] for p in epoch_losses])),\n                        \'mean_aux_loss\': float(np.mean([p[\'aux\'] for p in epoch_losses])),\n                        \'training_row_instances\': len(epoch_idx),\n                        \'expert_row_instances\': expert_instances,\n                        \'expert_row_fraction\': expert_instances / len(epoch_idx),\n                        \'outer_val_macro_auc\': None,\n                        \'selection_note\': \'outer labels not inspected during training\'})\n        atomic_save(checkpoint_path, {\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(),\n                                      \'scaler\': scaler.state_dict(), \'epoch\': epoch,\n                                      \'rng_state\': rng_state(batch_rng), \'seed\': seed,\n                                      \'history\': history, \'selection\': \'fixed_final_epoch\',\n                                      \'training_config\': training_config})\n\n    probs, attention = _predict(model, imgs_t, masks_t, val_idx, device,\n                                prediction_batch_size, use_amp)\n    selected_epoch = epochs - 1\n    outer_val_auc = _val_macro_auc(probs, y_expert, expert_mask, val_idx)\n    return probs, history, stopping_reason, attention, selected_epoch, outer_val_auc\n\n\ndef run_arm(arm_name, cache_dir, ids, gold_ids, gold_labels, report_source, policy, out_dir,\n            model_fn, k=5, epochs=3, lr=1e-3, seed=1400, batch_size=8, resume=True, fold_seed=1400,\n            patience=None, device=\'auto\', amp=None, prediction_batch_size=None,\n            backbone_lr=8e-6, weight_decay=0.02, crossfit_policy=False,\n            policy_bootstrap=1000, policy_seed=1400, folds=None,\n            expert_fraction=0.10):\n    """Trains one arm (\'baseline\' has an all-zero policy; \'auxiliary\' uses a\n    transfer-gated policy) across k grouped folds and writes an OOF prediction\n    file plus a receipt. Only studies in `gold_ids` ever contribute an expert-BCE\n    term or an OOF row scored against ground truth; report-only pool studies\n    still participate in training (auxiliary term only) but are not evaluated."""\n    from cache import load_batch\n\n    start = time.time()\n    out_dir = Path(out_dir)\n    resolved_device = resolve_device(device)\n    use_amp = resolved_device.type == \'cuda\' if amp is None else bool(amp and resolved_device.type == \'cuda\')\n    imgs, masks = load_batch(cache_dir, ids)\n\n    fold_info = assign_folds(ids, k=k, seed=fold_seed, priority_ids=gold_ids)\n    fold_of = np.array([fold_info[\'fold_assignment\'][u] for u in ids])\n    selected_folds = list(range(k)) if folds is None else sorted(set(map(int, folds)))\n    if not selected_folds or any(fold < 0 or fold >= k for fold in selected_folds):\n        raise ValueError(f\'folds must be a non-empty subset of 0..{k - 1}\')\n\n    oof = np.full((len(ids), len(TARGETS)), np.nan)\n    oof_covered = np.zeros(len(ids), dtype=bool)\n    fold_histories = {}\n    fold_policies = {}\n    fold_policy_audits = {}\n    checkpoints = {}\n    for fold in selected_folds:\n        val_idx = np.flatnonzero(fold_of == fold)\n        train_idx = np.flatnonzero(fold_of != fold)\n        if len(val_idx) == 0 or len(train_idx) == 0:\n            raise ValueError(f\'fold {fold} has an empty split; reduce k or add studies\')\n        fold_policy = policy\n        if crossfit_policy:\n            fold_policy, fold_audit = crossfit_transfer_policy(\n                ids, train_idx, gold_labels, report_source,\n                bootstrap=policy_bootstrap, seed=policy_seed + fold * 100)\n            fold_policy_audits[str(fold)] = fold_audit\n        fold_policies[str(fold)] = fold_policy\n        y_expert, expert_mask, y_aux, aux_mask, aux_weight = build_supervision(\n            ids, gold_labels, report_source, fold_policy)\n        ckpt = out_dir / \'checkpoints\' / f\'{arm_name}_fold{fold}.pt\'\n        probs, history, reason, _, selected_epoch, outer_val_auc = train_one_fold(\n            model_fn, imgs, masks, y_expert, expert_mask, y_aux, aux_mask, aux_weight,\n            train_idx, val_idx, epochs, lr, seed, ckpt, batch_size, resume, patience,\n            resolved_device.type, use_amp, prediction_batch_size, backbone_lr, weight_decay,\n            expert_fraction)\n        oof[val_idx] = probs\n        oof_covered[val_idx] = True\n        fold_histories[fold] = history\n        checkpoints[str(fold)] = {\n            \'path\': str(ckpt), \'sha256\': digest_file(ckpt), \'stopping_reason\': reason,\n            \'selected_epoch\': selected_epoch, \'outer_val_macro_auc_after_selection\': outer_val_auc,\n            \'selection\': \'fixed final epoch; outer labels were not used for selection\',\n            \'epochs_run\': len(history), \'epochs_requested\': epochs,\n        }\n        if resolved_device.type == \'cuda\':\n            torch.cuda.empty_cache()\n\n    common_receipt = {\n        \'arm\': arm_name, \'git_commit\': git_commit(), \'seed\': seed, \'fold_seed\': fold_seed,\n        \'k\': k, \'selected_folds\': selected_folds,\n        \'fold_assignment_sha256\': hashlib.sha256(json.dumps(\n            fold_info[\'fold_assignment\'], sort_keys=True).encode()).hexdigest(),\n        \'epochs_requested\': epochs, \'patience\': patience,\n        \'head_lr\': lr, \'backbone_lr\': backbone_lr, \'weight_decay\': weight_decay,\n        \'batch_size\': batch_size, \'expert_fraction\': expert_fraction,\n        \'n_studies_total\': len(ids), \'n_studies_scored\': len(gold_ids),\n        \'policy\': policy, \'policy_selection\': (\'cross-fitted on training-fold expert cases\'\n                                              if crossfit_policy else \'fixed predeclared policy\'),\n        \'policy_bootstrap\': policy_bootstrap if crossfit_policy else None,\n        \'policy_seed\': policy_seed if crossfit_policy else None,\n        \'fold_policies\': fold_policies, \'fold_policy_audits\': fold_policy_audits,\n        \'fold_grouping_caveat\': fold_info[\'grouping_caveat\'],\n        \'checkpoints\': checkpoints, \'fold_histories\': fold_histories,\n        \'epoch_selection\': (\'fixed final epoch declared before training; outer-fold expert labels \'\n                            \'were scored once afterward and never selected a checkpoint\'),\n        \'runtime_seconds\': time.time() - start, \'device\': str(resolved_device),\n        \'amp_fp16\': use_amp,\n    }\n\n    if not oof_covered.all():\n        partial_frame = pd.DataFrame(oof[oof_covered], columns=TARGETS)\n        partial_frame.insert(0, UID, np.asarray(ids)[oof_covered])\n        partial_path = out_dir / f\'{arm_name}_partial_oof.csv\'\n        partial_path.parent.mkdir(parents=True, exist_ok=True)\n        with partial_path.open(\'x\', encoding=\'utf-8\') as f:\n            partial_frame.to_csv(f, index=False)\n        receipt = {\n            **common_receipt,\n            \'status\': \'COMPLETE_FOLD_SHARD_NOT_COMPLETE_ARM\',\n            \'partial_oof_path\': str(partial_path),\n            \'partial_oof_sha256\': digest_file(partial_path),\n            \'partial_oof_rows\': len(partial_frame),\n        }\n        write_receipt(out_dir / f\'{arm_name}_partial_receipt.json\', receipt)\n        return partial_frame, receipt\n\n    all_frame = pd.DataFrame(oof, columns=TARGETS)\n    all_frame.insert(0, UID, ids)\n    all_oof_path = out_dir / f\'{arm_name}_all_oof.csv\'\n    all_oof_path.parent.mkdir(parents=True, exist_ok=True)\n    with all_oof_path.open(\'x\', encoding=\'utf-8\') as f:\n        all_frame.to_csv(f, index=False)\n\n    gold_only = [i for i, u in enumerate(ids) if u in set(gold_ids)]\n    frame = pd.DataFrame(oof[gold_only], columns=TARGETS)\n    frame.insert(0, UID, [ids[i] for i in gold_only])\n    oof_path = out_dir / f\'{arm_name}_oof.csv\'\n    oof_path.parent.mkdir(parents=True, exist_ok=True)\n    with oof_path.open(\'x\', encoding=\'utf-8\') as f:\n        frame.to_csv(f, index=False)\n\n    receipt = {\n        **common_receipt,\n        \'status\': \'COMPLETE_ARM\',\n        \'n_studies_scored\': len(gold_only),\n        \'gold_oof_sha256\': digest_file(oof_path),\n        \'all_oof_sha256\': digest_file(all_oof_path),\n        \'all_oof_scope\': (\'Includes cross-fold predictions for report-only pool studies so the \'\n                          \'incremental frozen-ensemble blend can be evaluated at n≈4,349. Weak \'\n                          \'labels remain development evidence, not expert truth.\'),\n    }\n    write_receipt(out_dir / f\'{arm_name}_receipt.json\', receipt)\n    return frame, receipt\n', encoding='utf-8')
(_src / 'compare_oof.py').write_text('"""Automated, paired comparison of the baseline-arm vs. auxiliary-arm OOF\npredictions against expert gold labels. Same statistical standard as v14\'s\ndiagnostics.py: grouped paired bootstrap, no unpaired comparisons, explicit\nscope caveats rather than a bare number.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import rankdata\n\nfrom contract import TARGETS, UID\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef align(frame, ids):\n    if frame[UID].isna().any() or frame[UID].duplicated().any():\n        raise ValueError(\'Missing or duplicate study IDs\')\n    if set(frame[UID]) != set(ids):\n        raise ValueError(\'Prediction coverage does not exactly match the requested study IDs; \'\n                         \'partial scoring is prohibited\')\n    return frame.set_index(UID).loc[list(ids)].reset_index()\n\n\ndef auc(y, p):\n    valid = np.isfinite(y)\n    y, p = y[valid], p[valid]\n    pos, neg = np.sum(y == 1), np.sum(y == 0)\n    if not pos or not neg:\n        return np.nan\n    return float((rankdata(p)[y == 1].sum() - pos * (pos + 1) / 2) / (pos * neg))\n\n\ndef aucs(y, p):\n    return np.array([auc(y[:, j], p[:, j]) for j in range(len(TARGETS))])\n\n\ndef compare(labels_path, baseline_oof_path, auxiliary_oof_path, bootstrap=2000, seed=1400):\n    labels = pd.read_csv(labels_path, dtype={UID: str})\n    baseline = pd.read_csv(baseline_oof_path, dtype={UID: str})\n    auxiliary = pd.read_csv(auxiliary_oof_path, dtype={UID: str})\n    ids = labels[UID].tolist()\n    baseline = align(baseline, ids)\n    auxiliary = align(auxiliary, ids)\n    y = labels.set_index(UID).loc[ids, TARGETS].to_numpy(float)\n    b = baseline[TARGETS].to_numpy(float)\n    a = auxiliary[TARGETS].to_numpy(float)\n    if not (np.isin(y, [0.0, 1.0])).all():\n        raise ValueError(\'Labels must be strictly binary for this comparison\')\n\n    ba, aa = aucs(y, b), aucs(y, a)\n    rng = np.random.default_rng(seed)\n    n = len(ids)\n    draws = []\n    for _ in range(bootstrap):\n        idx = rng.integers(0, n, n)\n        d = aucs(y[idx], a[idx]) - aucs(y[idx], b[idx])\n        if np.isfinite(d).all():\n            draws.append(d)\n    if len(draws) < 0.5 * bootstrap:\n        raise ValueError(\'Too many resamples lacked both classes; cohort too small/imbalanced for this comparison\')\n    draws = np.asarray(draws)\n    macro_ci = np.quantile(draws.mean(axis=1), [.025, .975])\n    target_ci = np.quantile(draws, [.025, .975], axis=0)\n\n    return {\n        \'scope\': (\'Paired comparison on the SAME 58-study gold cohort that also selected the \'\n                  \'auxiliary-loss policy (see labels.py). This is exploratory, not independent \'\n                  \'confirmation, and not a hidden-test-set estimate.\'),\n        \'cohort_studies\': n,\n        \'baseline_macro_auc\': float(ba.mean()), \'auxiliary_macro_auc\': float(aa.mean()),\n        \'macro_delta\': float((aa - ba).mean()), \'macro_delta_ci95\': macro_ci.tolist(),\n        \'bootstrap_seed\': seed, \'valid_replicates\': len(draws), \'requested_replicates\': bootstrap,\n        \'targets\': {t: {\'baseline_auc\': float(ba[j]), \'auxiliary_auc\': float(aa[j]),\n                        \'delta\': float(aa[j] - ba[j]), \'delta_ci95\': target_ci[:, j].tolist()}\n                   for j, t in enumerate(TARGETS)},\n        \'input_sha256\': {\'labels\': digest(labels_path), \'baseline_oof\': digest(baseline_oof_path),\n                         \'auxiliary_oof\': digest(auxiliary_oof_path)},\n    }\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--labels\', type=Path, required=True)\n    p.add_argument(\'--baseline-oof\', type=Path, required=True)\n    p.add_argument(\'--auxiliary-oof\', type=Path, required=True)\n    p.add_argument(\'--bootstrap\', type=int, default=2000)\n    p.add_argument(\'--seed\', type=int, default=1400)\n    p.add_argument(\'--out\', type=Path, required=True)\n    args = p.parse_args()\n    result = compare(args.labels, args.baseline_oof, args.auxiliary_oof, args.bootstrap, args.seed)\n    with args.out.open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(result, f, indent=2)\n        f.write(\'\\n\')\n    print(f"macro delta {result[\'macro_delta\']:+.6f}, CI95 {result[\'macro_delta_ci95\']}")\n    for t in TARGETS:\n        d = result[\'targets\'][t]\n        print(f"  {t:18s} base={d[\'baseline_auc\']:.4f} aux={d[\'auxiliary_auc\']:.4f} "\n              f"delta={d[\'delta\']:+.4f} ci95={d[\'delta_ci95\']}")\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
(_src / 'family_c_gate.py').write_text('"""Family C reproducibility gate: only proceed to Family C once Family A shows a\nreproducible, same-direction gain across two independently seeded training runs.\nThis is a project-management gate, not statistical proof (the plan is explicit\nabout that distinction) -- with a 58-study cohort, requiring the bootstrap CI\nlower bound to clear zero on both seeds would almost never pass regardless of\nwhether the underlying effect is real, so the gate checks sign- and\nmagnitude-consistency across seeds instead of a per-seed significance test.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\n\n\ndef evaluate_gate(seed_reports, min_macro_gain=0.001):\n    """seed_reports: list of >=2 compare_oof.py-style report dicts, one per\n    training seed, each comparing the same two arms. PASS requires every seed\'s\n    macro_delta to exceed min_macro_gain AND all seeds to agree in sign."""\n    if len(seed_reports) < 2:\n        raise ValueError(\'Need at least two seeds to evaluate reproducibility\')\n    deltas = [r[\'macro_delta\'] for r in seed_reports]\n    signs = {1 if d > 0 else (-1 if d < 0 else 0) for d in deltas}\n    all_above_threshold = all(d > min_macro_gain for d in deltas)\n    consistent_sign = len(signs) == 1 and 0 not in signs\n    passed = all_above_threshold and consistent_sign\n\n    per_target_agreement = {}\n    targets = seed_reports[0][\'targets\'].keys()\n    for t in targets:\n        target_deltas = [r[\'targets\'][t][\'delta\'] for r in seed_reports]\n        target_signs = {1 if d > 0 else (-1 if d < 0 else 0) for d in target_deltas}\n        per_target_agreement[t] = {\n            \'deltas_by_seed\': target_deltas,\n            \'sign_agreement\': len(target_signs) == 1 and 0 not in target_signs,\n        }\n    agreeing_targets = sum(v[\'sign_agreement\'] for v in per_target_agreement.values())\n\n    return {\n        \'decision\': \'PASS_PROCEED_TO_FAMILY_C\' if passed else \'FAIL_DO_NOT_START_FAMILY_C\',\n        \'reason\': (\'all seeds exceed the minimum gain with consistent sign\' if passed else\n                   \'macro deltas disagree in sign or direction across seeds\' if not consistent_sign else\n                   f\'not every seed exceeded the minimum gain of {min_macro_gain}\'),\n        \'macro_deltas_by_seed\': deltas,\n        \'min_macro_gain_required\': min_macro_gain,\n        \'targets_agreeing_in_sign\': agreeing_targets, \'targets_total\': len(per_target_agreement),\n        \'per_target\': per_target_agreement,\n        \'caveat\': (\'Project-management gate, not statistical proof: 58 studies do not support a \'\n                  \'per-seed significance requirement. A PASS means the observed direction was \'\n                  \'reproduced, not that the effect is confirmed at conventional significance.\'),\n    }\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--seed-reports\', type=Path, nargs=\'+\', required=True,\n                   help=\'Two or more compare_oof.py JSON outputs, one per training seed\')\n    p.add_argument(\'--min-macro-gain\', type=float, default=0.001,\n                   help=\'Required gain in both seeds; default matches the only measured useful \'\n                        \'diverse-family OOF effect in this repository\')\n    p.add_argument(\'--out\', type=Path, required=True)\n    args = p.parse_args()\n    reports = [json.loads(path.read_text(encoding=\'utf-8\')) for path in args.seed_reports]\n    result = evaluate_gate(reports, args.min_macro_gain)\n    with args.out.open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(result, f, indent=2)\n        f.write(\'\\n\')\n    print(result[\'decision\'])\n    print(result[\'reason\'])\n    print(f"targets agreeing in sign: {result[\'targets_agreeing_in_sign\']}/{result[\'targets_total\']}")\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
(_src / 'merge_shards.py').write_text('"""Merge independently executed Family A fold shards into one exact OOF arm.\n\nEvery shard is checked against the deterministic fold map, input hashes, model\nconfiguration, seed, and study coverage. A missing, duplicated, or foreign fold\nis fatal; this script never fills gaps or silently averages overlapping rows.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom contract import TARGETS, UID\nfrom folds import assign_folds\n\n\nCONFIG_KEYS = (\'arm\', \'seed\', \'fold_seed\', \'k\', \'epochs_requested\', \'head_lr\',\n               \'backbone_lr\', \'weight_decay\', \'batch_size\', \'policy_selection\',\n               \'policy_bootstrap\', \'policy_seed\', \'expert_fraction\',\n               \'fold_assignment_sha256\')\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef read_ids(path, name):\n    frame = pd.read_csv(path, dtype={UID: str})\n    if UID not in frame or frame[UID].isna().any() or frame[UID].duplicated().any():\n        raise ValueError(f\'{name} must contain unique, nonmissing {UID} values\')\n    return frame[UID].tolist()\n\n\ndef resolve_prediction(receipt_path, receipt):\n    recorded = Path(receipt[\'partial_oof_path\'])\n    candidates = [recorded, Path(receipt_path).parent / recorded.name]\n    prediction = next((path for path in candidates if path.is_file()), None)\n    if prediction is None:\n        raise FileNotFoundError(f\'Prediction file for {receipt_path} was not found\')\n    if digest(prediction) != receipt[\'partial_oof_sha256\']:\n        raise ValueError(f\'Prediction hash differs from receipt: {prediction}\')\n    return prediction\n\n\ndef merge_partial_receipts(receipt_paths, ids_path, gold_ids_path, out_dir):\n    receipt_paths = [Path(path) for path in receipt_paths]\n    if not receipt_paths:\n        raise ValueError(\'At least one shard receipt is required\')\n    ids = read_ids(ids_path, \'all-study table\')\n    gold_ids = read_ids(gold_ids_path, \'gold-study table\')\n    if not set(gold_ids).issubset(set(ids)):\n        raise ValueError(\'Gold IDs are not a subset of all study IDs\')\n    receipts = [json.loads(path.read_text(encoding=\'utf-8\')) for path in receipt_paths]\n    if any(receipt.get(\'status\') != \'COMPLETE_FOLD_SHARD_NOT_COMPLETE_ARM\'\n           for receipt in receipts):\n        raise ValueError(\'Every input must be a completed partial-arm receipt\')\n    expected_config = {key: receipts[0].get(key) for key in CONFIG_KEYS}\n    for receipt in receipts[1:]:\n        actual = {key: receipt.get(key) for key in CONFIG_KEYS}\n        if actual != expected_config:\n            raise ValueError(\'Shard model/fold configuration mismatch\')\n\n    k = int(expected_config[\'k\'])\n    fold_info = assign_folds(ids, k=k, seed=int(expected_config[\'fold_seed\']),\n                             priority_ids=gold_ids)\n    expected_fold_hash = hashlib.sha256(json.dumps(\n        fold_info[\'fold_assignment\'], sort_keys=True).encode()).hexdigest()\n    if expected_fold_hash != expected_config[\'fold_assignment_sha256\']:\n        raise ValueError(\'Shard fold map does not match supplied study IDs\')\n\n    seen_folds = set()\n    pieces = []\n    source_records = []\n    for receipt_path, receipt in zip(receipt_paths, receipts):\n        selected = set(map(int, receipt[\'selected_folds\']))\n        if not selected or seen_folds.intersection(selected):\n            raise ValueError(\'Shard folds are empty or overlap\')\n        prediction_path = resolve_prediction(receipt_path, receipt)\n        frame = pd.read_csv(prediction_path, dtype={UID: str})\n        if UID not in frame or frame[UID].isna().any() or frame[UID].duplicated().any():\n            raise ValueError(f\'Invalid prediction identities in {prediction_path}\')\n        expected_ids = {uid for uid in ids\n                        if fold_info[\'fold_assignment\'][uid] in selected}\n        if set(frame[UID]) != expected_ids:\n            raise ValueError(f\'Prediction coverage does not equal declared folds: {prediction_path}\')\n        values = frame[TARGETS].to_numpy(float)\n        if not (np.isfinite(values) & (values >= 0) & (values <= 1)).all():\n            raise ValueError(f\'Invalid probabilities in {prediction_path}\')\n        pieces.append(frame[[UID] + TARGETS])\n        source_records.append({\'receipt\': str(receipt_path),\n                               \'receipt_sha256\': digest(receipt_path),\n                               \'prediction\': str(prediction_path),\n                               \'prediction_sha256\': digest(prediction_path),\n                               \'folds\': sorted(selected)})\n        seen_folds.update(selected)\n    if seen_folds != set(range(k)):\n        raise ValueError(f\'Incomplete folds: got {sorted(seen_folds)}, need 0..{k - 1}\')\n\n    merged = pd.concat(pieces, ignore_index=True)\n    if merged[UID].duplicated().any() or set(merged[UID]) != set(ids):\n        raise ValueError(\'Merged shards do not cover every study exactly once\')\n    merged = merged.set_index(UID).loc[ids].reset_index()\n    gold = merged.set_index(UID).loc[gold_ids].reset_index()\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    arm = expected_config[\'arm\']\n    all_path = out_dir / f\'{arm}_all_oof.csv\'\n    gold_path = out_dir / f\'{arm}_oof.csv\'\n    with all_path.open(\'x\', encoding=\'utf-8\') as stream:\n        merged.to_csv(stream, index=False)\n    with gold_path.open(\'x\', encoding=\'utf-8\') as stream:\n        gold.to_csv(stream, index=False)\n    result = {\n        \'status\': \'COMPLETE_ARM_MERGED_FROM_SHARDS\', **expected_config,\n        \'selected_folds\': sorted(seen_folds), \'n_studies_total\': len(merged),\n        \'n_studies_scored\': len(gold), \'sources\': source_records,\n        \'all_oof_sha256\': digest(all_path), \'gold_oof_sha256\': digest(gold_path),\n    }\n    receipt_out = out_dir / f\'{arm}_receipt.json\'\n    with receipt_out.open(\'x\', encoding=\'utf-8\') as stream:\n        json.dump(result, stream, indent=2, allow_nan=False)\n        stream.write(\'\\n\')\n    return merged, gold, result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--receipts\', type=Path, nargs=\'+\', required=True)\n    parser.add_argument(\'--ids\', type=Path, required=True)\n    parser.add_argument(\'--gold-ids\', type=Path, required=True)\n    parser.add_argument(\'--out-dir\', type=Path, required=True)\n    args = parser.parse_args()\n    _, _, result = merge_partial_receipts(args.receipts, args.ids, args.gold_ids, args.out_dir)\n    print(f"merged {result[\'arm\']} seed {result[\'seed\']} across {result[\'k\']} folds")\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
(_src / 'evaluate_submission_gain.py').write_text('"""Paired OOF evaluation of adding Family A to a frozen existing ensemble.\n\nThis estimates development-cohort discrimination change. It never calls the\nresult an actual leaderboard gain: public/private transport remains unknown\nuntil Kaggle scores a frozen submission.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import rankdata\n\nfrom contract import TARGETS, UID\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef align(frame, ids, name):\n    if UID not in frame or frame[UID].isna().any() or frame[UID].duplicated().any():\n        raise ValueError(f\'{name}: missing or duplicate study IDs\')\n    if set(frame[UID]) != set(ids):\n        raise ValueError(f\'{name}: coverage differs from labels; partial scoring is prohibited\')\n    values = frame.set_index(UID).loc[ids, TARGETS].to_numpy(float)\n    if not (np.isfinite(values) & (values >= 0) & (values <= 1)).all():\n        raise ValueError(f\'{name}: predictions must be finite probabilities in [0,1]\')\n    return values\n\n\ndef labels_from_frame(frame, source_kind):\n    if UID not in frame or frame[UID].isna().any() or frame[UID].duplicated().any():\n        raise ValueError(\'labels: missing or duplicate study IDs\')\n    raw = frame[TARGETS].to_numpy(float)\n    if source_kind == \'binary\':\n        if not (np.isnan(raw) | (raw == 0) | (raw == 1)).all():\n            raise ValueError(\'binary labels must be 0, 1, or missing\')\n        return raw\n    if source_kind == \'report-soft\':\n        if not (np.isfinite(raw) & (raw >= 0) & (raw <= 1)).all():\n            raise ValueError(\'report-soft labels must be finite in [0,1]\')\n        return np.where(raw == 0.5, np.nan, (raw > 0.5).astype(float))\n    raise ValueError("source_kind must be \'binary\' or \'report-soft\'")\n\n\ndef auc(y, prediction):\n    valid = np.isfinite(y) & np.isfinite(prediction)\n    y, prediction = y[valid], prediction[valid]\n    positive, negative = int((y == 1).sum()), int((y == 0).sum())\n    if not positive or not negative:\n        return np.nan\n    ranks = rankdata(prediction)\n    return float((ranks[y == 1].sum() - positive * (positive + 1) / 2) /\n                 (positive * negative))\n\n\ndef macro_auc(y, prediction):\n    values = np.array([auc(y[:, j], prediction[:, j]) for j in range(len(TARGETS))])\n    return float(np.nanmean(values)), values\n\n\ndef column_ranks(values):\n    return np.column_stack([rankdata(values[:, j]) / len(values) for j in range(values.shape[1])])\n\n\ndef evaluate(labels_path, baseline_path, family_paths, source_kind=\'binary\',\n             weights=(0.10,), bootstrap=5000, seed=1500, public_baseline=0.935):\n    labels = pd.read_csv(labels_path, dtype={UID: str})\n    ids = labels[UID].tolist()\n    y = labels_from_frame(labels, source_kind)\n    baseline = align(pd.read_csv(baseline_path, dtype={UID: str}), ids, \'baseline\')\n    family_by_seed = [align(pd.read_csv(path, dtype={UID: str}), ids, f\'family[{i}]\')\n                      for i, path in enumerate(family_paths)]\n    if len(family_by_seed) < 2:\n        raise ValueError(\'At least two independently seeded Family A OOF files are required\')\n\n    baseline_rank = column_ranks(baseline)\n    family_mean_rank = column_ranks(np.mean(family_by_seed, axis=0))\n    baseline_macro, baseline_targets = macro_auc(y, baseline_rank)\n    if not np.isfinite(baseline_targets).all():\n        raise ValueError(\'Every target needs both classes; silent target dropping is prohibited\')\n    rng = np.random.default_rng(seed)\n    results = {}\n    for weight in weights:\n        if not 0 < weight < 1:\n            raise ValueError(\'Every Family A blend weight must be strictly between 0 and 1\')\n        candidate = (1 - weight) * baseline_rank + weight * family_mean_rank\n        candidate_macro, candidate_targets = macro_auc(y, candidate)\n        if not np.isfinite(candidate_targets).all():\n            raise ValueError(\'Candidate is not scorable on every target\')\n        draws = []\n        for _ in range(bootstrap):\n            idx = rng.integers(0, len(ids), len(ids))\n            _, base_targets_draw = macro_auc(y[idx], baseline_rank[idx])\n            _, candidate_targets_draw = macro_auc(y[idx], candidate[idx])\n            if np.isfinite(base_targets_draw).all() and np.isfinite(candidate_targets_draw).all():\n                draws.append(float((candidate_targets_draw - base_targets_draw).mean()))\n        if len(draws) < bootstrap * 0.8:\n            raise ValueError(\'Too many bootstrap replicates were unscorable\')\n        draws = np.asarray(draws)\n        delta = candidate_macro - baseline_macro\n        ci = np.quantile(draws, [.025, .975]).tolist()\n        results[f\'{weight:.6f}\'] = {\n            \'family_weight\': float(weight),\n            \'baseline_macro_auc\': baseline_macro,\n            \'candidate_macro_auc\': candidate_macro,\n            \'macro_delta\': delta,\n            \'macro_delta_ci95\': ci,\n            \'bootstrap_standard_error\': float(draws.std(ddof=1)),\n            \'probability_better\': float((draws > 0).mean()),\n            \'conditional_projected_public_score\': public_baseline + delta,\n            \'conditional_projected_public_score_ci95\': [public_baseline + ci[0],\n                                                        public_baseline + ci[1]],\n            \'reaches_0_950_point_estimate\': bool(public_baseline + delta >= 0.950),\n            \'reaches_0_950_ci_lower\': bool(public_baseline + ci[0] >= 0.950),\n            \'targets\': {target: {\n                \'baseline_auc\': float(baseline_targets[j]),\n                \'candidate_auc\': float(candidate_targets[j]),\n                \'delta\': float(candidate_targets[j] - baseline_targets[j]),\n            } for j, target in enumerate(TARGETS)},\n        }\n\n    primary = results[f\'{float(weights[0]):.6f}\']\n    seed_deltas = []\n    for prediction in family_by_seed:\n        candidate = (1 - weights[0]) * baseline_rank + weights[0] * column_ranks(prediction)\n        candidate_macro, _ = macro_auc(y, candidate)\n        seed_deltas.append(candidate_macro - baseline_macro)\n    seed_sd = float(np.std(seed_deltas, ddof=1))\n    seed_se_mean = seed_sd / np.sqrt(len(seed_deltas))\n    bootstrap_se = primary[\'bootstrap_standard_error\']\n    combined_se = float(np.sqrt(bootstrap_se ** 2 + seed_se_mean ** 2))\n    combined_delta_ci = [primary[\'macro_delta\'] - 1.96 * combined_se,\n                         primary[\'macro_delta\'] + 1.96 * combined_se]\n    rounding_half_unit = 0.0005\n    return {\n        \'scope\': (\'Paired OOF development estimate for a frozen rank blend. Conditional public-score \'\n                  \'projections assume one-for-one transfer and are not actual leaderboard scores.\'),\n        \'source_kind\': source_kind,\n        \'cohort_studies\': len(ids),\n        \'public_baseline_recorded\': public_baseline,\n        \'gain_required_for_0_950\': 0.950 - public_baseline,\n        \'primary_weight\': float(weights[0]),\n        \'primary_result\': primary,\n        \'all_predeclared_weights\': results,\n        \'primary_seed_deltas\': seed_deltas,\n        \'primary_seed_delta_sd_descriptive\': seed_sd,\n        \'error_propagation\': {\n            \'formula\': (\'SE_quantified ≈ sqrt(SE_paired_bootstrap^2 + \'\n                        \'(SD_training_seed/sqrt(n_seeds))^2). This omits transport shift.\'),\n            \'paired_bootstrap_se\': bootstrap_se,\n            \'training_seed_se_of_mean_descriptive\': float(seed_se_mean),\n            \'combined_se_approx\': combined_se,\n            \'combined_delta_ci95_normal_approx\': combined_delta_ci,\n            \'conditional_public_score_ci95_normal_approx\':\n                [public_baseline + combined_delta_ci[0], public_baseline + combined_delta_ci[1]],\n            \'conditional_public_score_ci95_including_baseline_display_rounding\':\n                [public_baseline - rounding_half_unit + combined_delta_ci[0],\n                 public_baseline + rounding_half_unit + combined_delta_ci[1]],\n            \'leaderboard_rounding\': {\n                \'assumption\': \'scores displayed to nearest 0.001\',\n                \'baseline_0_935_latent_interval\': [0.9345, 0.9355],\n                \'candidate_0_950_latent_interval\': [0.9495, 0.9505],\n                \'displayed_gain_0_015_latent_interval\': [0.0140, 0.0160],\n                \'latent_score_at_least_0_950_requires_displayed\': 0.951,\n            },\n            \'warning\': (\'The numerical interval is not a leaderboard prediction interval because \'\n                        \'validation-to-public transport and public/private shift are unquantified.\'),\n        },\n        \'error_budget\': {\n            \'cohort_sampling\': \'quantified by paired study bootstrap\',\n            \'training_seed\': (\'descriptive SD only; two seeds are insufficient for a reliable \'\n                              \'variance component\'),\n            \'policy_selection\': (\'not adjusted if the 58 gold cases selected the auxiliary policy\'),\n            \'validation_to_public_transport\': \'unquantified\',\n            \'public_to_private_shift\': \'unquantified\',\n            \'actual_proof\': \'requires a scored frozen Kaggle submission at or above 0.950\',\n        },\n        \'input_sha256\': {\n            \'labels\': digest(labels_path), \'baseline\': digest(baseline_path),\n            \'family_by_seed\': [digest(path) for path in family_paths],\n        },\n        \'bootstrap_seed\': seed,\n        \'bootstrap_requested_replicates\': bootstrap,\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--labels\', type=Path, required=True)\n    parser.add_argument(\'--baseline-oof\', type=Path, required=True)\n    parser.add_argument(\'--family-oof\', type=Path, nargs=\'+\', required=True)\n    parser.add_argument(\'--source-kind\', choices=[\'binary\', \'report-soft\'], default=\'binary\')\n    parser.add_argument(\'--weights\', type=float, nargs=\'+\', default=[0.10, 0.05, 0.15])\n    parser.add_argument(\'--bootstrap\', type=int, default=5000)\n    parser.add_argument(\'--seed\', type=int, default=1500)\n    parser.add_argument(\'--public-baseline\', type=float, default=0.935)\n    parser.add_argument(\'--out\', type=Path, required=True)\n    args = parser.parse_args()\n    result = evaluate(args.labels, args.baseline_oof, args.family_oof, args.source_kind,\n                      tuple(args.weights), args.bootstrap, args.seed, args.public_baseline)\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    with args.out.open(\'x\', encoding=\'utf-8\') as stream:\n        json.dump(result, stream, indent=2, allow_nan=False)\n        stream.write(\'\\n\')\n    primary = result[\'primary_result\']\n    print(f"primary blend delta {primary[\'macro_delta\']:+.6f}, "\n          f"CI95 {primary[\'macro_delta_ci95\']}")\n    print(f"conditional public score {primary[\'conditional_projected_public_score\']:.6f}; "\n          "actual leaderboard transfer remains unmeasured")\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
(_src / 'transfer_audit.py').write_text('"""Stage L0 target-transfer audit: does report-derived soft evidence rank-order\nexpert image labels well enough to justify capped auxiliary supervision?\n\nAUC answers a ranking question only -- it is not an agreement rate, not an error\nrate, and does not by itself certify calibration. This script deliberately keeps\nthose questions separate:\n\n  * auc_all_retain_silent   -- AUC treating every study\'s raw soft value as the\n                               prediction, including report-silent 0.5 cells.\n  * auc_addressed_only      -- AUC restricted to studies where the report source\n                               actually addressed the target (value != 0.5).\n  * silent_expert_prevalence -- among studies the report was silent on, what\n                               fraction are expert-positive. This tests whether\n                               silence behaves like an implicit negative; it does\n                               NOT establish that silence *causes* any AUC gap\n                               elsewhere, since silent and addressed cases can\n                               differ in other ways too.\n  * brier_addressed_only    -- mean squared error of the addressed soft values\n                               against expert truth: a calibration diagnostic,\n                               not a ranking one.\n\nEvery AUC is bootstrap-resampled at the study level with a percentile CI, and\nevery count is reported so wide intervals from small positive/negative counts\nare visible rather than hidden behind a single point estimate. This produces\nevidence for Stage L0 of the V15 label-supervision gate; it does not itself\ndecide which targets are promoted to training -- that call also needs the\ngrouped-fold validation the plan requires before any target/source pair is\nused as an auxiliary loss.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import rankdata\n\nUID = \'StudyInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nMIN_CLASS_COUNT_FOR_AUC = 3\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef require_unique_ids(frame, name):\n    if frame[UID].isna().any():\n        raise ValueError(f\'{name}: missing study IDs\')\n    if frame[UID].duplicated().any():\n        dupes = frame.loc[frame[UID].duplicated(), UID].unique()[:3].tolist()\n        raise ValueError(f\'{name}: duplicate study IDs, e.g. {dupes}\')\n\n\ndef load_aligned(gold_path, source_path):\n    gold = pd.read_csv(gold_path, dtype={UID: str})\n    source = pd.read_csv(source_path, dtype={UID: str})\n    require_unique_ids(gold, \'gold labels\')\n    require_unique_ids(source, \'report source\')\n    if not set(TARGETS).issubset(gold.columns) or not set(TARGETS).issubset(source.columns):\n        raise ValueError(\'Both tables must carry all twelve targets\')\n    missing = set(gold[UID]) - set(source[UID])\n    if missing:\n        raise ValueError(f\'{len(missing)} gold studies absent from report source, e.g. {sorted(missing)[:3]}\')\n    gold = gold.set_index(UID)\n    source = source.set_index(UID).loc[gold.index]\n    y = gold[TARGETS].to_numpy(float)\n    if not np.isin(y, [0.0, 1.0]).all():\n        raise ValueError(\'Gold labels must be strictly binary; this audit requires a fully labeled cohort\')\n    p = source[TARGETS].to_numpy(float)\n    if not (np.isfinite(p) & (p >= 0) & (p <= 1)).all():\n        raise ValueError(\'Report-derived values must be finite probabilities in [0, 1]\')\n    return gold.index.to_numpy(), y, p\n\n\ndef auc(y, p):\n    y, p = np.asarray(y, float), np.asarray(p, float)\n    valid = np.isfinite(y)\n    y, p = y[valid], p[valid]\n    pos, neg = int((y == 1).sum()), int((y == 0).sum())\n    if pos < MIN_CLASS_COUNT_FOR_AUC or neg < MIN_CLASS_COUNT_FOR_AUC:\n        return float(\'nan\')\n    return float((rankdata(p)[y == 1].sum() - pos * (pos + 1) / 2) / (pos * neg))\n\n\ndef brier(y, p):\n    y, p = np.asarray(y, float), np.asarray(p, float)\n    valid = np.isfinite(y)\n    if not valid.any():\n        return None\n    return float(np.mean((p[valid] - y[valid]) ** 2))\n\n\ndef brier_skill(y, p):\n    """Brier score is not comparable across targets of differing prevalence: a\n    target that is 90% negative scores well by predicting 0.1 everywhere. Report\n    skill relative to the prevalence-only baseline (always predict the base rate)\n    instead of the raw score, so 0 means no better than knowing prevalence alone\n    and 1 means perfect."""\n    y = np.asarray(y, float)\n    valid = np.isfinite(y)\n    if not valid.any():\n        return {\'brier\': None, \'prevalence_baseline_brier\': None, \'skill_score\': None}\n    y = y[valid]\n    prevalence = float(y.mean())\n    baseline_brier = float(np.mean((prevalence - y) ** 2))\n    model_brier = brier(y, p[valid] if hasattr(p, \'__len__\') else p)\n    if baseline_brier <= 1e-12:\n        return {\'brier\': model_brier, \'prevalence_baseline_brier\': baseline_brier, \'skill_score\': None}\n    return {\'brier\': model_brier, \'prevalence_baseline_brier\': baseline_brier,\n            \'skill_score\': float(1 - model_brier / baseline_brier)}\n\n\ndef bootstrap_auc(y, p, n=2000, seed=1400):\n    y, p = np.asarray(y, float), np.asarray(p, float)\n    point = auc(y, p)\n    if not np.isfinite(point):\n        return {\'point\': None, \'ci95\': None, \'valid_replicates\': 0, \'requested_replicates\': n,\n                \'note\': \'fewer than %d cases in one class; AUC undefined\' % MIN_CLASS_COUNT_FOR_AUC}\n    rng = np.random.default_rng(seed)\n    n_obs = len(y)\n    draws = []\n    for _ in range(n):\n        idx = rng.integers(0, n_obs, n_obs)\n        a = auc(y[idx], p[idx])\n        if np.isfinite(a):\n            draws.append(a)\n    result = {\'point\': point, \'valid_replicates\': len(draws), \'requested_replicates\': n}\n    if len(draws) < 0.5 * n:\n        result[\'ci95\'] = None\n        result[\'note\'] = \'fewer than half the resamples had both classes; CI unreliable at this n\'\n    else:\n        result[\'ci95\'] = np.quantile(draws, [.025, .975]).tolist()\n    return result\n\n\ndef audit_target(y, p, silent_value=0.5, bootstrap=2000, seed=1400):\n    addressed = p != silent_value\n    pos_addr = int((y[addressed] == 1).sum())\n    neg_addr = int((y[addressed] == 0).sum())\n    pos_silent = int((y[~addressed] == 1).sum())\n    neg_silent = int((y[~addressed] == 0).sum())\n    return {\n        \'n_total\': int(len(y)),\n        \'n_addressed\': int(addressed.sum()),\n        \'n_silent\': int((~addressed).sum()),\n        \'addressed_positive\': pos_addr, \'addressed_negative\': neg_addr,\n        \'silent_positive\': pos_silent, \'silent_negative\': neg_silent,\n        \'addressed_expert_prevalence\': (pos_addr / (pos_addr + neg_addr)) if (pos_addr + neg_addr) else None,\n        \'silent_expert_prevalence\': (pos_silent / (pos_silent + neg_silent)) if (pos_silent + neg_silent) else None,\n        \'auc_all_retain_silent\': bootstrap_auc(y, p, bootstrap, seed),\n        \'auc_addressed_only\': bootstrap_auc(y[addressed], p[addressed], bootstrap, seed),\n        \'brier_addressed_only\': brier_skill(y[addressed], p[addressed]),\n        \'small_sample_warning\': min(pos_addr, neg_addr) < 10,\n    }\n\n\ndef run(gold_path, source_path, bootstrap=2000, seed=1400):\n    ids, y, p = load_aligned(gold_path, source_path)\n    targets = {t: audit_target(y[:, j], p[:, j], bootstrap=bootstrap, seed=seed) for j, t in enumerate(TARGETS)}\n    return {\n        \'scope\': (\'Stage L0 transfer audit. AUC is a ranking diagnostic, not an agreement or \'\n                  \'error rate. Point estimates from <=58 studies carry wide uncertainty; treat \'\n                  \'ci95=None as "cannot be estimated reliably at this n", not as zero evidence. \'\n                  \'Brier is reported only as a skill score against the per-target prevalence \'\n                  \'baseline, since raw Brier is not comparable across targets of differing \'\n                  \'prevalence. NON-INDEPENDENCE: if this audit\\\'s own output is used to choose \'\n                  \'which target/source pairs get auxiliary-loss weight, any later evaluation of \'\n                  \'that policy on this same 58-study cohort is exploratory, not independent \'\n                  \'confirmation -- the cohort selected the policy it is then used to score.\'),\n        \'cohort_studies\': int(len(ids)),\n        \'input_sha256\': {\'gold_labels\': digest(gold_path), \'report_source\': digest(source_path)},\n        \'bootstrap_seed\': seed, \'bootstrap_requested_replicates\': bootstrap,\n        \'targets\': targets,\n        \'decision_note\': (\'This audit alone does not select target/source pairs for training. \'\n                          \'Promotion additionally requires the grouped-fold auxiliary-loss \'\n                          \'ablation in Run 4 of the V15 plan to show a cross-fitted expert-label \'\n                          \'gain, not just a favorable audit AUC.\'),\n    }\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    p.add_argument(\'--gold-labels\', type=Path, required=True)\n    p.add_argument(\'--report-source\', type=Path, required=True)\n    p.add_argument(\'--bootstrap\', type=int, default=2000)\n    p.add_argument(\'--seed\', type=int, default=1400)\n    p.add_argument(\'--out\', type=Path, required=True)\n    args = p.parse_args()\n    result = run(args.gold_labels, args.report_source, args.bootstrap, args.seed)\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    with args.out.open(\'x\', encoding=\'utf-8\') as stream:\n        json.dump(result, stream, indent=2, allow_nan=False)\n        stream.write(\'\\n\')\n    header = f\'{"target":18s} {"n_addr":>6s} {"pos/neg":>9s} {"auc_addr":>10s} {"ci95":>22s} {"brier_skill":>11s} {"small_n":>7s}\'\n    print(header)\n    for t in TARGETS:\n        d = result[\'targets\'][t]\n        a = d[\'auc_addressed_only\']\n        ci = f"[{a[\'ci95\'][0]:.3f},{a[\'ci95\'][1]:.3f}]" if a[\'ci95\'] else \'n/a\'\n        auc_str = f"{a[\'point\']:.4f}" if a[\'point\'] is not None else \'n/a\'\n        skill = d[\'brier_addressed_only\'][\'skill_score\']\n        brier_str = f"{skill:.4f}" if skill is not None else \'n/a\'\n        print(f"{t:18s} {d[\'n_addressed\']:6d} {d[\'addressed_positive\']}/{d[\'addressed_negative\']:>7d} "\n              f"{auc_str:>10s} {ci:>22s} {brier_str:>11s} {\'yes\' if d[\'small_sample_warning\'] else \'no\':>7s}")\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
sys.path.insert(0, '/kaggle/working/family_a_src')
print('family_a modules written to', _src)

In [ ]:

import cache as family_a_cache

test_series = pd.read_csv(ROOT / 'test_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
plane_map = dict(zip(test_series['SeriesInstanceUID'], test_series['Anatomical_Plane']))
hte = annotate(walk('test_series'))
slot_map = pick_slots(hte, plane_map)
study_ids = pd.read_csv(ROOT / 'test.csv', dtype={'StudyInstanceUID': str})['StudyInstanceUID'].tolist()
log(f'cache build: {len(study_ids)} studies, {N_SLOT} slots')

CACHE_OUT = Path('/kaggle/working/family_a_gold_cache')
built = []
for sid in study_ids:
    chosen = slot_map.get(sid, {})
    imgs = np.zeros((N_SLOT, GROUP, IMG, IMG), np.uint8)
    mask = np.zeros(N_SLOT, dtype=bool)
    for k, (name, plane, fluid, fs) in enumerate(SLOTS):
        if name not in chosen:
            continue
        rec = dict(chosen[name])
        rec['ordered'], _ok = order_slices(rec)
        tile = read_slot(rec)
        if tile is None:
            continue
        imgs[k] = tile.numpy()
        mask[k] = True
    if not mask.any():
        log(f'WARNING: {sid} has no usable slot, skipping (no expert/aux signal possible for it)')
        continue
    family_a_cache.write_study(CACHE_OUT, sid, imgs, mask)
    built.append(sid)
    if len(built) % 200 == 0:
        log(f'cached {len(built)}/{len(study_ids)}')

family_a_cache.build_manifest(CACHE_OUT, built)
pd.Series(built, name='StudyInstanceUID').to_csv(CACHE_OUT / 'cached_ids.csv', index=False)
print(f'DONE caching {len(built)}/{len(study_ids)} studies to {CACHE_OUT}. Cache build only, DO NOT SUBMIT.', flush=True)
